# 🧠 AI Personal Finance Coach — Application Server
### `server.ipynb` — the single backend brain

Runs the complete backend on **http://localhost:5050**, notebook-safely (uvicorn in a dedicated
daemon thread with its own event loop — no `asyncio.run()` conflicts, ever). Serves the five
frontend files and the full REST API.

| Subsystem | What it does |
|---|---|
| **Storage** | Atomic JSON persistence — accounts, sessions, per-day transaction files, 23:59 daily summaries (with startup catch-up), notifications |
| **Accounts** | Register / login on the (username, password, country) triple · salted PBKDF2 hashing · token sessions |
| **Transactions** | Manual entry + OCR drafts · validation · bilingual auto-categorization (rules → payee memory → LLM) |
| **Intelligence** | Spending analysis, month-end projection with uncertainty band, anomaly detection, budget advisor, Financial Health Score, What-If simulator, subscription detector |
| **OCR** | EasyOCR (fa+en, lazy-loaded) → normalization → LLM structured extraction → regex fallback |
| **Bitcoin** | Loads `data/btc/model.keras` + scalers from `bitcoin.ipynb` · live prices (CryptoCompare → CoinGecko → dataset fallback) · 10-day forecast + UP/DOWN/HOLD signal |
| **LLM** | Grounded chat with tool access to the last 30 days of JSON, streamed with `Searching…` status events · Whisper STT · notification phrasing |
| **Notifications** | Deterministic rules fire → LLM phrases (template fallback) → bilingual, cooldown-guarded |
| **Resilience** | Every AI dependency (LLM key, EasyOCR, BTC artifacts, price APIs) is optional — the server degrades gracefully and reports capability status at `/api/status` |

**Run order:** run all cells top to bottom. The server starts in the *LAUNCH* cell and keeps running
while the kernel is alive. Re-running the LAUNCH cell restarts it cleanly.

**API map** (all under `/api`, JSON envelope `{ok, data, error}`):
`POST auth/register · POST auth/login · POST auth/logout · GET|PUT profile · POST tx · GET tx ·
DELETE tx/{date}/{id} · POST ocr · POST stt · POST chat (NDJSON stream) · GET insights · GET report ·
GET health-score · POST whatif · GET subscriptions · GET notifications · GET notifications/count ·
POST notifications/read · GET btc/history · GET btc/forecast · GET summary/daily · GET status ·
POST dev/seed` — plus `/`, `/index.html`, `/AI.html`, `/bitcoin.html`, `/notification.html`, `/style.css`.

In [1]:
import os, io, re, json, math, time, uuid, base64, pickle, random, hashlib, secrets, threading, tempfile, warnings
from datetime import datetime, date, timedelta, timezone
from zoneinfo import ZoneInfo
from collections import defaultdict

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import requests
from fastapi import FastAPI, Header, UploadFile, File, Request
from fastapi.responses import JSONResponse, FileResponse, StreamingResponse, Response
from fastapi.middleware.cors import CORSMiddleware
import uvicorn

CAPS = {}
try:
    from openai import OpenAI
    CAPS["openai_sdk"] = True
except ImportError:
    CAPS["openai_sdk"] = False
try:
    import tensorflow as tf
    from tensorflow import keras
    CAPS["tensorflow"] = True
except ImportError:
    CAPS["tensorflow"] = False
try:
    from PIL import Image, ImageOps
    CAPS["pillow"] = True
except ImportError:
    CAPS["pillow"] = False
try:
    import importlib.util
    CAPS["easyocr"] = importlib.util.find_spec("easyocr") is not None   # imported lazily later (heavy)
except Exception:
    CAPS["easyocr"] = False

GAPGPT_API_KEY = os.environ.get("GAPGPT_API_KEY", "") or "sk-TR4mcAch3tvWzeOPQiMbAghIu2eoeeSTJ7WM1hwejVP93khf"

CONFIG = {
    "port": 5050,
    "app_name": os.environ.get("PFC_APP_NAME", "Fiscora"),
    "logo_path": os.environ.get("PFC_LOGO_PATH", "assets/logo.svg"),
    "base_dir": os.getcwd(),
    "data_dir": "data",
    "llm_base_url": "https://api.gapgpt.app/v1",
    "llm_model": "gpt-5.4",
    "stt_model": "whisper-1",
    "llm_temperature": 0.3,
    "history_window_days": 30,
    "session_file": "data/sessions.json",
    "users_file": "data/users.json",
    "btc_artifacts_dir": "data/btc",
    "btc_dataset_fallback": "datasets/btc/btc_test.csv",
    "max_upload_mb": 12,
    "pbkdf2_iterations": 310_000,
    "sweep_interval_s": 1800,
    "btc_refresh_s": 3600,
}

DATA_DIR = CONFIG["data_dir"]
os.makedirs(os.path.join(DATA_DIR, "users"), exist_ok=True)
os.makedirs(CONFIG["btc_artifacts_dir"], exist_ok=True)

CATEGORIES = ["food_dining", "groceries", "transport", "housing_utilities", "health",
              "entertainment", "shopping", "education", "subscriptions", "transfers",
              "income", "other"]
CATEGORY_LABELS = {
    "food_dining":       {"en": "Food & Dining",      "fa": "غذا و رستوران"},
    "groceries":         {"en": "Groceries",          "fa": "خواربار"},
    "transport":         {"en": "Transport",          "fa": "حمل‌ونقل"},
    "housing_utilities": {"en": "Housing & Utilities","fa": "مسکن و قبوض"},
    "health":            {"en": "Health",             "fa": "سلامت و درمان"},
    "entertainment":     {"en": "Entertainment",      "fa": "سرگرمی"},
    "shopping":          {"en": "Shopping",           "fa": "خرید"},
    "education":         {"en": "Education",          "fa": "آموزش"},
    "subscriptions":     {"en": "Subscriptions",      "fa": "اشتراک‌ها"},
    "transfers":         {"en": "Transfers",          "fa": "انتقال وجه"},
    "income":            {"en": "Income",             "fa": "درآمد"},
    "other":             {"en": "Other",              "fa": "سایر"},
}
TX_TYPES = ["purchase", "card_to_card", "online_payment", "subscription",
            "income", "transfer_out", "other"]

COUNTRY_META = {
    "IR": {"timezone": "Asia/Tehran",       "currency": "TOMAN", "language": "fa",
           "calendar": "jalali",   "week_start_dow": 5, "digest_dow": 4},   # week starts Saturday; digest Friday
    "US": {"timezone": "America/New_York",  "currency": "USD",   "language": "en",
           "calendar": "gregorian","week_start_dow": 6, "digest_dow": 6},   # digest Sunday
}
CURRENCIES = {"TOMAN", "USD"}

ERRORS = {
    "auth_required":        {"en": "Please log in first.",                          "fa": "ابتدا وارد حساب خود شوید."},
    "invalid_credentials":  {"en": "Username, password and country do not match.",  "fa": "نام کاربری، رمز عبور و کشور مطابقت ندارند."},
    "account_exists_login": {"en": "This account already exists — please log in.",  "fa": "این حساب قبلاً ساخته شده است — لطفاً وارد شوید."},
    "username_taken":       {"en": "This username is already taken.",               "fa": "این نام کاربری قبلاً گرفته شده است."},
    "invalid_input":        {"en": "Invalid input.",                                "fa": "ورودی نامعتبر است."},
    "not_found":            {"en": "Not found.",                                    "fa": "موردی یافت نشد."},
    "ocr_unavailable":      {"en": "OCR engine is not installed on the server (pip install easyocr).", "fa": "موتور OCR روی سرور نصب نیست (pip install easyocr)."},
    "llm_unavailable":      {"en": "AI service is not configured (set GAPGPT_API_KEY).", "fa": "سرویس هوش مصنوعی پیکربندی نشده است (GAPGPT_API_KEY)."},
    "btc_unavailable":      {"en": "Bitcoin model artifacts not found — run bitcoin.ipynb first.", "fa": "مدل بیت‌کوین یافت نشد — ابتدا bitcoin.ipynb را اجرا کنید."},
    "file_too_large":       {"en": "Uploaded file is too large.",                   "fa": "حجم فایل ارسالی بیش از حد مجاز است."},
    "server_error":         {"en": "Internal server error.",                        "fa": "خطای داخلی سرور."},
}

import sys
print("=== capability check ===")
print(f"  kernel       {sys.executable}")
for k, v in CAPS.items():
    print(f"  {k:<12} {'OK' if v else 'missing (optional)'}")
print(f"  llm_key      {'configured' if GAPGPT_API_KEY else 'NOT SET — AI features run in fallback mode'}")
if not CAPS["tensorflow"]:
    print("\n  ⚠️  TensorFlow is not installed in THIS kernel's environment — Bitcoin forecasts")
    print(f"     will be disabled. Fix:  {sys.executable} -m pip install tensorflow")
    print("     then restart the kernel and run all cells again.")
print(f"\nproject root : {CONFIG['base_dir']}")
print(f"data dir     : {os.path.abspath(DATA_DIR)}")

=== capability check ===
  kernel       /Users/sam/Desktop/aiolearn/venv/bin/python
  openai_sdk   OK
  tensorflow   OK
  pillow       OK
  easyocr      OK
  llm_key      configured

project root : /Users/sam/Desktop/aiolearn/My_projects/innoverse_mansory_edition
data dir     : /Users/sam/Desktop/aiolearn/My_projects/innoverse_mansory_edition/data


## 2 · Storage layer — atomic JSON, Jalali dates, accounts, sessions

Design rules enforced here and used by every subsystem:

- **Atomic writes** — every JSON write goes to a temp file in the same directory, then `os.replace`.
  A killed kernel can never leave a half-written file. A global re-entrant lock serializes writers.
- **Money is always `{amount, currency}`** — never a bare number. Iranian amounts are integer
  **Toman**; US amounts are decimal USD. No implicit cross-currency arithmetic anywhere.
- **User-local time** — each user lives in their country's timezone (`Asia/Tehran` / `America/New_York`);
  daily files are keyed by the user-local date, and Iranian dates carry a Jalali twin field.
- **Auth** — salted PBKDF2-SHA256 (310k iterations). Login requires the exact
  (username, password, country) triple per the competition spec. Sessions persist across kernel
  restarts in `data/sessions.json`.

In [2]:
_IO_LOCK = threading.RLock()


def read_json(path, default=None):
    with _IO_LOCK:
        if not os.path.exists(path):
            return default
        with open(path, "r", encoding="utf-8") as fh:
            return json.load(fh)


def write_json_atomic(path, obj):
    with _IO_LOCK:
        d = os.path.dirname(path) or "."
        os.makedirs(d, exist_ok=True)
        fd, tmp = tempfile.mkstemp(dir=d, suffix=".tmp")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as fh:
                json.dump(obj, fh, ensure_ascii=False, indent=2)
            os.replace(tmp, path)
        finally:
            if os.path.exists(tmp):
                os.unlink(tmp)


def gregorian_to_jalali(gy: int, gm: int, gd: int) -> str:
    g_dm = [0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334]
    gy2 = gy - 1600
    days = 365 * gy2 + (gy2 + 3) // 4 - (gy2 + 99) // 100 + (gy2 + 399) // 400 - 80 + gd + g_dm[gm - 1]
    if gm > 2 and ((gy % 4 == 0 and gy % 100 != 0) or gy % 400 == 0):
        days += 1
    jy = 979 + 33 * (days // 12053)
    days %= 12053
    jy += 4 * (days // 1461)
    days %= 1461
    if days > 365:
        jy += (days - 1) // 365
        days = (days - 1) % 365
    jm = 1 + days // 31 if days < 186 else 7 + (days - 186) // 30
    jd = 1 + (days % 31 if days < 186 else (days - 186) % 30)
    return f"{jy:04d}-{jm:02d}-{jd:02d}"


def jalali_of(iso_date: str) -> str:
    y, m, d = (int(p) for p in iso_date.split("-"))
    return gregorian_to_jalali(y, m, d)


def hash_password(password: str, salt: str = None) -> str:
    salt = salt or secrets.token_hex(16)
    dk = hashlib.pbkdf2_hmac("sha256", password.encode("utf-8"), bytes.fromhex(salt),
                             CONFIG["pbkdf2_iterations"])
    return f"pbkdf2_sha256${CONFIG['pbkdf2_iterations']}${salt}${dk.hex()}"


def verify_password(password: str, stored: str) -> bool:
    try:
        _, _, salt, _ = stored.split("$")
        return secrets.compare_digest(hash_password(password, salt), stored)
    except (ValueError, AttributeError):
        return False


def user_dir(user_id):     return os.path.join(DATA_DIR, "users", user_id)
def profile_path(user_id): return os.path.join(user_dir(user_id), "profile.json")
def tx_dir(user_id):       return os.path.join(user_dir(user_id), "transactions")
def summary_dir(user_id):  return os.path.join(user_dir(user_id), "summaries")
def notif_path(user_id):   return os.path.join(user_dir(user_id), "notifications.json")
def payee_mem_path(user_id): return os.path.join(user_dir(user_id), "payee_categories.json")
def receipts_dir(user_id): return os.path.join(user_dir(user_id), "receipts")


class AccountService:
    """users.json registry + per-user directory bootstrap."""

    def _load(self):
        return read_json(CONFIG["users_file"], {"schema_version": 1, "users": []})

    def register(self, username: str, password: str, country: str):
        username = (username or "").strip()
        if not username or not password or country not in COUNTRY_META:
            return None, "invalid_input"
        db = self._load()
        for u in db["users"]:
            if u["username"] == username:
                if verify_password(password, u["password_hash"]) and u["country"] == country:
                    return None, "account_exists_login"
                return None, "username_taken"
        meta = COUNTRY_META[country]
        now = datetime.now(ZoneInfo(meta["timezone"])).isoformat(timespec="seconds")
        user = {"user_id": "u_" + uuid.uuid4().hex[:8], "username": username,
                "password_hash": hash_password(password), "country": country,
                "created_at": now, "last_login_at": now}
        db["users"].append(user)
        write_json_atomic(CONFIG["users_file"], db)

        for d in (tx_dir(user["user_id"]), summary_dir(user["user_id"]), receipts_dir(user["user_id"])):
            os.makedirs(d, exist_ok=True)
        write_json_atomic(profile_path(user["user_id"]), {
            "schema_version": 1, "user_id": user["user_id"], "username": username,
            "country": country, "language": meta["language"], "currency": meta["currency"],
            "calendar": meta["calendar"],
            "balance": {"amount": 0, "currency": meta["currency"],
                        "updated_at": now},
            "monthly_budget": {"amount": 0, "currency": meta["currency"]},
            "category_budgets": {},
            "preferences": {"notifications_enabled": True, "btc_alerts": True,
                            "currency_display": "TOMAN" if country == "IR" else "USD"},
        })
        write_json_atomic(notif_path(user["user_id"]), {"schema_version": 1, "notifications": []})
        return user, None

    def login(self, username: str, password: str, country: str):
        db = self._load()
        for u in db["users"]:
            if (u["username"] == (username or "").strip() and u["country"] == country
                    and verify_password(password, u["password_hash"])):
                u["last_login_at"] = datetime.now(
                    ZoneInfo(COUNTRY_META[country]["timezone"])).isoformat(timespec="seconds")
                write_json_atomic(CONFIG["users_file"], db)
                return u, None
        return None, "invalid_credentials"

    def all_user_ids(self):
        return [u["user_id"] for u in self._load()["users"]]

    def get_profile(self, user_id):
        return read_json(profile_path(user_id))

    def save_profile(self, user_id, profile):
        write_json_atomic(profile_path(user_id), profile)


class SessionStore:
    """Token → user_id map, persisted so sessions survive kernel restarts."""

    def __init__(self):
        self._sessions = read_json(CONFIG["session_file"], {})

    def create(self, user_id):
        token = secrets.token_hex(24)
        self._sessions[token] = {"user_id": user_id,
                                 "created_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        write_json_atomic(CONFIG["session_file"], self._sessions)
        return token

    def user_id(self, token):
        s = self._sessions.get(token or "")
        return s["user_id"] if s else None

    def destroy(self, token):
        if token in self._sessions:
            del self._sessions[token]
            write_json_atomic(CONFIG["session_file"], self._sessions)


accounts = AccountService()
sessions = SessionStore()


def user_now(profile) -> datetime:
    return datetime.now(ZoneInfo(COUNTRY_META[profile["country"]]["timezone"]))


print("storage layer ready — atomic I/O, PBKDF2 auth, Jalali calendar "
      f"(today in Tehran: {jalali_of(datetime.now(ZoneInfo('Asia/Tehran')).date().isoformat())})")

storage layer ready — atomic I/O, PBKDF2 auth, Jalali calendar (today in Tehran: 1405-05-27)


In [3]:
class TransactionStore:

    def day_path(self, user_id, iso_date):
        return os.path.join(tx_dir(user_id), f"{iso_date}.json")

    def summary_path(self, user_id, iso_date):
        return os.path.join(summary_dir(user_id), f"{iso_date}.summary.json")

    def _empty_day(self, user_id, iso_date, now_iso):
        return {"schema_version": 1, "file_type": "daily_transactions", "date": iso_date,
                "date_jalali": jalali_of(iso_date), "created_at": now_iso,
                "last_updated_at": now_iso, "user_id": user_id, "transactions": []}

    def validate(self, profile, payload):
        """Returns (clean_tx, error_code). Strict: bad money never enters storage."""
        if not isinstance(payload, dict):
            return None, "invalid_input"
        tx_type = payload.get("type", "purchase")
        if tx_type not in TX_TYPES:
            return None, "invalid_input"
        currency = payload.get("currency", profile["currency"])
        if currency not in CURRENCIES:
            return None, "invalid_input"
        try:
            amount = float(payload.get("amount", 0))
        except (TypeError, ValueError):
            return None, "invalid_input"
        if not math.isfinite(amount) or amount <= 0 or amount > 1e15:
            return None, "invalid_input"
        if currency == "TOMAN":
            amount = int(round(amount))

        ts_raw = payload.get("timestamp")
        try:
            if ts_raw:
                ts = datetime.fromisoformat(ts_raw.replace("Z", "+00:00"))
            else:
                ts = user_now(profile)
        except (TypeError, ValueError, AttributeError):
            return None, "invalid_input"
        if ts.tzinfo is None:
            ts = ts.replace(tzinfo=ZoneInfo(COUNTRY_META[profile["country"]]["timezone"]))

        category = payload.get("category") or None
        if category is not None and category not in CATEGORIES:
            return None, "invalid_input"

        items, clean_items = payload.get("items") or [], []
        for it in items[:100]:
            try:
                qty = max(1, int(it.get("quantity", 1)))
                unit = float(it["unit_price"]["amount"] if isinstance(it.get("unit_price"), dict)
                             else it.get("unit_price", 0))
                name = str(it.get("name", "")).strip()[:120]
            except (TypeError, ValueError, KeyError):
                continue
            if name and math.isfinite(unit) and unit > 0:
                clean_items.append({"name": name, "quantity": qty,
                                    "unit_price": {"amount": unit, "currency": currency},
                                    "line_total": {"amount": round(unit * qty, 2), "currency": currency}})

        raw_tags = payload.get("tags") or []
        clean_tags = [str(x).strip()[:24] for x in list(raw_tags)[:8] if str(x).strip()]

        return {"type": tx_type, "currency": currency, "amount": amount, "timestamp": ts,
                "payee": str(payload.get("payee", "")).strip()[:120],
                "note": str(payload.get("note", "")).strip()[:500],
                "category": category, "items": clean_items, "tags": clean_tags,
                "source": payload.get("source", "manual"),
                "ocr": payload.get("ocr") if isinstance(payload.get("ocr"), dict) else None}, None

    def add(self, user_id, profile, payload):
        clean, err = self.validate(profile, payload)
        if err:
            return None, err
        if clean["category"] is None:
            cat, conf, method = categorizer.classify(user_id, clean["payee"], clean["note"],
                                                     clean["items"], clean["type"])
        else:
            cat, conf, method = clean["category"], 1.0, "user"
        categorizer.remember(user_id, clean["payee"], cat)

        ts = clean["timestamp"]
        iso_date = ts.date().isoformat()
        now_iso = user_now(profile).isoformat(timespec="seconds")
        tx = {"transaction_id": f"tx_{ts:%Y%m%d}_{uuid.uuid4().hex[:6]}",
              "timestamp": ts.isoformat(timespec="seconds"),
              "type": clean["type"], "source": clean["source"],
              "payee": clean["payee"], "note": clean["note"],
              "currency": clean["currency"],
              "total": {"amount": clean["amount"], "currency": clean["currency"]},
              "category": cat, "category_confidence": round(conf, 2), "category_method": method,
              "items": clean["items"], "tags": clean.get("tags", [])}
        if clean["ocr"]:
            tx["ocr"] = clean["ocr"]

        day = read_json(self.day_path(user_id, iso_date)) or self._empty_day(user_id, iso_date, now_iso)
        day["transactions"].append(tx)
        day["last_updated_at"] = now_iso
        write_json_atomic(self.day_path(user_id, iso_date), day)

        if clean["currency"] == profile["balance"]["currency"]:
            delta = clean["amount"] if clean["type"] == "income" else -clean["amount"]
            profile["balance"]["amount"] = round(profile["balance"]["amount"] + delta, 2)
            profile["balance"]["updated_at"] = now_iso
            accounts.save_profile(user_id, profile)
        return tx, None

    def delete(self, user_id, profile, iso_date, tx_id):
        day = read_json(self.day_path(user_id, iso_date))
        if not day:
            return "not_found"
        keep, removed = [], None
        for t in day["transactions"]:
            (keep.append(t) if t["transaction_id"] != tx_id else (removed := t))
        if removed is None:
            return "not_found"
        day["transactions"] = keep
        day["last_updated_at"] = user_now(profile).isoformat(timespec="seconds")
        write_json_atomic(self.day_path(user_id, iso_date), day)
        if removed["currency"] == profile["balance"]["currency"]:
            delta = -removed["total"]["amount"] if removed["type"] == "income" else removed["total"]["amount"]
            profile["balance"]["amount"] = round(profile["balance"]["amount"] + delta, 2)
            accounts.save_profile(user_id, profile)
        return None

    def days_in_range(self, user_id, start_date, end_date):
        out, d = [], start_date
        while d <= end_date:
            doc = read_json(self.day_path(user_id, d.isoformat()))
            if doc:
                out.append(doc)
            d += timedelta(days=1)
        return out

    def list(self, user_id, profile, start=None, end=None, category=None, query=None, limit=200):
        today = user_now(profile).date()
        start_d = date.fromisoformat(start) if start else today - timedelta(days=30)
        end_d = date.fromisoformat(end) if end else today
        rows = []
        for doc in self.days_in_range(user_id, start_d, end_d):
            for t in doc["transactions"]:
                if category and t["category"] != category:
                    continue
                if query:
                    hay = f"{t['payee']} {t['note']} " + " ".join(i["name"] for i in t.get("items", []))
                    if query.lower() not in hay.lower():
                        continue
                rows.append(t)
        rows.sort(key=lambda t: t["timestamp"], reverse=True)
        return rows[: max(1, min(int(limit), 1000))]

    def last_n_day_files(self, user_id, profile, n=30):
        today = user_now(profile).date()
        return self.days_in_range(user_id, today - timedelta(days=n - 1), today)

    def build_summary(self, user_id, profile, iso_date, mode="scheduled"):
        day = read_json(self.day_path(user_id, iso_date))
        txs = day["transactions"] if day else []
        cur = profile["currency"]
        spent = sum(t["total"]["amount"] for t in txs if t["type"] != "income" and t["currency"] == cur)
        income = sum(t["total"]["amount"] for t in txs if t["type"] == "income" and t["currency"] == cur)
        by_cat, by_type = defaultdict(lambda: {"amount": 0, "count": 0}), defaultdict(int)
        largest = None
        for t in txs:
            by_type[t["type"]] += 1
            if t["type"] != "income" and t["currency"] == cur:
                by_cat[t["category"]]["amount"] = round(by_cat[t["category"]]["amount"] + t["total"]["amount"], 2)
                by_cat[t["category"]]["count"] += 1
                if largest is None or t["total"]["amount"] > largest["total"]["amount"]:
                    largest = t
        summary = {"schema_version": 1, "file_type": "daily_summary", "date": iso_date,
                   "date_jalali": jalali_of(iso_date),
                   "generated_at": user_now(profile).isoformat(timespec="seconds"),
                   "generation_mode": mode,
                   "totals": {"spent": {"amount": round(spent, 2), "currency": cur},
                              "income": {"amount": round(income, 2), "currency": cur},
                              "transaction_count": len(txs),
                              "item_count": sum(len(t.get("items", [])) for t in txs)},
                   "by_category": {k: {"amount": v["amount"], "currency": cur, "count": v["count"]}
                                   for k, v in sorted(by_cat.items(), key=lambda kv: -kv[1]["amount"])},
                   "by_type": dict(by_type),
                   "largest_transaction_id": largest["transaction_id"] if largest else None,
                   "balance_end_of_day": dict(profile["balance"]),
                   "raw_transactions": txs}
        write_json_atomic(self.summary_path(user_id, iso_date), summary)
        return summary

    def catch_up_summaries(self, user_id, profile):
        """On startup: create summaries for any past day that has transactions but no summary
        (the notebook is not a daemon — it may have been off at 23:59)."""
        created = 0
        today_iso = user_now(profile).date().isoformat()
        if not os.path.isdir(tx_dir(user_id)):
            return 0
        for fname in sorted(os.listdir(tx_dir(user_id))):
            if not fname.endswith(".json"):
                continue
            iso_date = fname[:-5]
            if iso_date < today_iso and not os.path.exists(self.summary_path(user_id, iso_date)):
                self.build_summary(user_id, profile, iso_date, mode="catchup")
                created += 1
        return created


store = TransactionStore()
print("transaction store ready — per-day JSON, validation, summaries, catch-up")

transaction store ready — per-day JSON, validation, summaries, catch-up


## 3 · Auto-categorization — rules → payee memory → LLM

Three tiers, cheapest first; the tier that decided is recorded on every transaction
(`category_method: rules | memory | llm | user`) so every categorization is auditable:

1. **Rules** — curated bilingual keyword maps over the 12 canonical categories. Instant, offline,
   explainable. Persian text is normalized first (Arabic ي/ك → Persian ی/ک, Persian/Arabic-Indic
   digits → ASCII) so matching is robust to keyboard variants.
2. **Payee memory** — once a payee is categorized (by any tier, or corrected by the user), it is
   remembered per user. The app *feels* smarter every day at zero cost.
3. **LLM** — unknown payees go to the model with the closed category enum (temperature 0); the result
   is cached back into payee memory. Without an API key this tier is skipped and `other` is used —
   the app never blocks on AI availability.

In [4]:
_FA_TRANS = str.maketrans({"ي": "ی", "ك": "ک", "ۀ": "ه", "ة": "ه",
                           **{chr(0x06F0 + i): str(i) for i in range(10)},    # ۰-۹
                           **{chr(0x0660 + i): str(i) for i in range(10)}})   # ٠-٩


def normalize_text(s: str) -> str:
    return (s or "").translate(_FA_TRANS).lower().strip()


CATEGORY_KEYWORDS = {
    "food_dining": ["restaurant", "cafe", "coffee", "pizza", "burger", "kebab", "snappfood", "doordash",
                    "grubhub", "mcdonald", "kfc", "starbucks", "رستوران", "کافه", "قهوه", "پیتزا",
                    "برگر", "کباب", "اسنپ فود", "اسنپ‌فود", "فست فود", "چلوکباب", "ساندویچ", "نانوایی", "نان"],
    "groceries": ["grocery", "supermarket", "market", "walmart", "costco", "trader joe", "kroger",
                  "safeway", "سوپرمارکت", "هایپر", "رفاه", "شهروند", "افق کوروش", "میوه", "سبزی",
                  "خواربار", "لبنیات", "قصابی", "پروتئین"],
    "transport": ["uber", "lyft", "taxi", "metro", "bus", "train", "gas station", "fuel", "parking",
                  "اسنپ", "تپسی", "تاکسی", "مترو", "اتوبوس", "بنزین", "پارکینگ", "عوارض", "قطار"],
    "housing_utilities": ["rent", "mortgage", "electric", "electricity", "water bill", "internet bill",
                          "اجاره", "رهن", "قبض", "برق", "آب", "گاز", "اینترنت", "شارژ ساختمان"],
    "health": ["pharmacy", "doctor", "dentist", "hospital", "clinic", "cvs", "walgreens",
               "داروخانه", "دکتر", "پزشک", "دندانپزشک", "بیمارستان", "درمانگاه", "آزمایشگاه", "دارو"],
    "entertainment": ["cinema", "movie", "game", "concert", "spotify premium", "playstation",
                      "سینما", "فیلم", "کنسرت", "بازی", "تفریح", "شهربازی", "بلیط"],
    "shopping": ["amazon", "ebay", "mall", "clothing", "shoes", "target", "best buy", "nike", "zara",
                 "دیجی کالا", "دیجی‌کالا", "پوشاک", "کفش", "لباس", "بازار", "فروشگاه", "کیف"],
    "education": ["course", "udemy", "coursera", "book", "tuition", "university",
                  "دوره", "کلاس", "کتاب", "شهریه", "دانشگاه", "آموزش", "مدرسه"],
    "subscriptions": ["netflix", "spotify", "subscription", "membership", "icloud", "youtube premium",
                      "اشتراک", "فیلیمو", "نماوا", "عضویت"],
    "transfers": ["transfer", "card to card", "paypal transfer", "venmo", "zelle", "wire",
                  "کارت به کارت", "انتقال", "حواله", "واریز به"],
    "income": ["salary", "payroll", "deposit", "income", "refund",
               "حقوق", "واریز حقوق", "درآمد", "برگشت وجه", "پاداش"],
}


class CategorizationService:

    def _memory(self, user_id):
        return read_json(payee_mem_path(user_id), {})

    def remember(self, user_id, payee, category):
        key = normalize_text(payee)
        if not key or category not in CATEGORIES:
            return
        mem = self._memory(user_id)
        if mem.get(key) != category:
            mem[key] = category
            write_json_atomic(payee_mem_path(user_id), mem)

    def classify(self, user_id, payee, note, items, tx_type):
        """→ (category, confidence, method)"""
        if tx_type == "income":
            return "income", 1.0, "rules"
        if tx_type in ("card_to_card", "transfer_out"):
            hay0 = normalize_text(f"{payee} {note}")
            if not any(kw in hay0 for cat in ("food_dining", "shopping", "housing_utilities")
                       for kw in map(normalize_text, CATEGORY_KEYWORDS[cat])):
                return "transfers", 0.9, "rules"

        mem_hit = self._memory(user_id).get(normalize_text(payee))
        if mem_hit:
            return mem_hit, 0.95, "memory"

        hay = normalize_text(" ".join([payee or "", note or ""] +
                                      [i["name"] for i in (items or [])]))
        best, best_hits = None, 0
        for cat, kws in CATEGORY_KEYWORDS.items():
            hits = sum(1 for kw in kws if normalize_text(kw) in hay)
            if hits > best_hits:
                best, best_hits = cat, hits
        if best:
            return best, min(0.6 + 0.15 * best_hits, 0.9), "rules"

        if llm.available and hay.strip():
            cat = llm.classify_category(hay)
            if cat in CATEGORIES:
                return cat, 0.8, "llm"
        return "other", 0.3, "rules"


categorizer = CategorizationService()
print(f"categorizer ready — {sum(len(v) for v in CATEGORY_KEYWORDS.values())} bilingual keywords, "
      "payee memory, LLM tier")

categorizer ready — 164 bilingual keywords, payee memory, LLM tier


## 4 · Financial analysis engine — deterministic, explainable, bilingual

Every insight the app shows (and everything the LLM is allowed to say about the user's money) is
computed **here, in Python** — the LLM only phrases what this engine measured. Each insight is a
*fact object*: type, severity, the numbers, a deterministic bilingual message, and the evidence
transaction ids — so the UI can let the user drill into *why*.

- **Month-end projection** — spend-to-date + weekday-weighted daily burn (trailing 28 days), with a
  **P20–P80 uncertainty band** from bootstrap resampling of observed daily spends. Honest error bars
  beat a fancy model at personal-data scale.
- **Trend deltas** — current 30 days vs. the previous 30, guarded by a minimum-share floor so
  "entertainment up 300%" can never be triggered by two coffees.
- **Anomaly detection** — z-scores on weekly category spend vs. the user's own baseline, plus
  large-single-transaction detection vs. the user's own P95.

In [5]:
def fmt_money(amount, currency, lang):
    a = f"{amount:,.0f}" if currency == "TOMAN" else f"{amount:,.2f}"
    if lang == "fa":
        return f"{a} تومان" if currency == "TOMAN" else f"{a} دلار"
    return f"{a} Toman" if currency == "TOMAN" else f"${a}"


class AnalysisEngine:

    def daily_series(self, user_id, profile, days):
        """date → {'total': x, 'by_category': {cat: y}, 'tx_ids': {cat: [...]}} for spend only,
        account currency only. Days with no file count as zero-spend days."""
        cur = profile["currency"]
        today = user_now(profile).date()
        start = today - timedelta(days=days - 1)
        series = {(start + timedelta(days=i)).isoformat():
                  {"total": 0.0, "by_category": defaultdict(float), "tx_ids": defaultdict(list)}
                  for i in range(days)}
        for doc in store.days_in_range(user_id, start, today):
            slot = series[doc["date"]]
            for t in doc["transactions"]:
                if t["type"] == "income" or t["currency"] != cur:
                    continue
                slot["total"] += t["total"]["amount"]
                slot["by_category"][t["category"]] += t["total"]["amount"]
                slot["tx_ids"][t["category"]].append(t["transaction_id"])
        return series

    def month_to_date(self, user_id, profile):
        cur = profile["currency"]
        today = user_now(profile).date()
        first = today.replace(day=1)
        spent = income = 0.0
        by_cat = defaultdict(float)
        for doc in store.days_in_range(user_id, first, today):
            for t in doc["transactions"]:
                if t["currency"] != cur:
                    continue
                if t["type"] == "income":
                    income += t["total"]["amount"]
                else:
                    spent += t["total"]["amount"]
                    by_cat[t["category"]] += t["total"]["amount"]
        return {"month": today.strftime("%Y-%m"), "first_day": first, "today": today,
                "spent": round(spent, 2), "income": round(income, 2),
                "by_category": {k: round(v, 2) for k, v in
                                sorted(by_cat.items(), key=lambda kv: -kv[1])}}

    def projection(self, user_id, profile):
        mtd = self.month_to_date(user_id, profile)
        today = mtd["today"]
        days_in_month = ((today.replace(day=28) + timedelta(days=4)).replace(day=1) - timedelta(days=1)).day
        remaining = [today + timedelta(days=i) for i in range(1, days_in_month - today.day + 1)]

        hist = self.daily_series(user_id, profile, 28)
        totals = {d: v["total"] for d, v in hist.items()}
        by_dow = defaultdict(list)
        for d, amt in totals.items():
            by_dow[date.fromisoformat(d).weekday()].append(amt)
        overall = list(totals.values())
        overall_mean = float(np.mean(overall)) if overall else 0.0
        dow_mean = {dw: float(np.mean(v)) for dw, v in by_dow.items()}

        expected_rest = sum(dow_mean.get(d.weekday(), overall_mean) for d in remaining)
        projected = round(mtd["spent"] + expected_rest, 2)

        rng, samples = np.random.default_rng(7), []
        pool = np.array(overall) if overall else np.array([0.0])
        for _ in range(300):
            samples.append(mtd["spent"] + float(rng.choice(pool, size=len(remaining), replace=True).sum())
                           if remaining else mtd["spent"])
        p20, p80 = (float(np.percentile(samples, 20)), float(np.percentile(samples, 80))) \
            if samples else (projected, projected)

        budget = profile["monthly_budget"]["amount"] or 0
        over_share = float(np.mean([s > budget for s in samples])) if (budget and samples) else 0.0
        return {"month": mtd["month"], "spent_to_date": mtd["spent"],
                "days_elapsed": today.day, "days_in_month": days_in_month,
                "projected_month_end": projected,
                "band_p20": round(p20, 2), "band_p80": round(p80, 2),
                "daily_burn_28d": round(overall_mean, 2),
                "budget": budget, "overrun_probability": round(over_share, 2),
                "currency": profile["currency"], "bootstrap_samples": samples}

    def category_deltas(self, user_id, profile):
        hist = self.daily_series(user_id, profile, 60)
        dates = sorted(hist.keys())
        prev_d, cur_d = dates[:30], dates[30:]
        prev, cur = defaultdict(float), defaultdict(float)
        for d in prev_d:
            for c, v in hist[d]["by_category"].items():
                prev[c] += v
        for d in cur_d:
            for c, v in hist[d]["by_category"].items():
                cur[c] += v
        total_cur = sum(cur.values()) or 1.0
        out = []
        for c in set(prev) | set(cur):
            p, q = prev.get(c, 0.0), cur.get(c, 0.0)
            share = q / total_cur
            if share < 0.03 and q < total_cur * 0.03:
                continue
            delta_pct = ((q - p) / p * 100) if p > 0 else (100.0 if q > 0 else 0.0)
            out.append({"category": c, "previous_30d": round(p, 2), "current_30d": round(q, 2),
                        "delta_pct": round(delta_pct, 1), "share_pct": round(share * 100, 1)})
        return sorted(out, key=lambda r: -abs(r["delta_pct"]))

    def anomalies(self, user_id, profile):
        found = []
        hist = self.daily_series(user_id, profile, 56)          # 8 weeks
        dates = sorted(hist.keys())
        weeks = [dates[i:i + 7] for i in range(0, 56, 7)]
        cat_weekly = defaultdict(list)
        for wk in weeks:
            wsum = defaultdict(float)
            for d in wk:
                for c, v in hist[d]["by_category"].items():
                    wsum[c] += v
            for c in CATEGORIES:
                cat_weekly[c].append(wsum.get(c, 0.0))
        for c, series in cat_weekly.items():
            base, last = series[:-1], series[-1]
            mu, sd = float(np.mean(base)), float(np.std(base))
            if sd > 0 and last > 0 and (last - mu) / sd >= 2.0 and last > mu * 1.5:
                found.append({"kind": "category_spike", "category": c,
                              "this_week": round(last, 2), "baseline_week": round(mu, 2),
                              "z": round((last - mu) / sd, 1),
                              "tx_ids": [i for d in weeks[-1] for i in hist[d]["tx_ids"].get(c, [])]})
                
        amounts = [t["total"]["amount"] for t in store.list(
            user_id, profile, start=(user_now(profile).date() - timedelta(days=90)).isoformat(),
            limit=1000) if t["type"] != "income"]
        if len(amounts) >= 12:
            p95 = float(np.percentile(amounts, 95))
            for t in store.list(user_id, profile,
                                start=(user_now(profile).date() - timedelta(days=2)).isoformat(),
                                limit=100):
                if t["type"] != "income" and t["total"]["amount"] > p95 * 1.5:
                    found.append({"kind": "large_transaction", "category": t["category"],
                                  "amount": t["total"]["amount"], "p95": round(p95, 2),
                                  "payee": t["payee"], "tx_ids": [t["transaction_id"]]})
        return found

    def insights(self, user_id, profile):
        lang_cur, lang = profile["currency"], profile["language"]
        cat_lbl = lambda c, lg: CATEGORY_LABELS.get(c, CATEGORY_LABELS["other"])[lg]
        out = []
        for d in self.category_deltas(user_id, profile)[:6]:
            if abs(d["delta_pct"]) < 15:
                continue
            up = d["delta_pct"] > 0
            out.append({"insight_id": "in_" + uuid.uuid4().hex[:6], "type": "trend",
                        "severity": "warning" if (up and d["delta_pct"] >= 40) else "info",
                        "category": d["category"], "facts": d,
                        "message": {
                            "en": f"{cat_lbl(d['category'],'en')} spending "
                                  f"{'rose' if up else 'fell'} {abs(d['delta_pct']):.0f}% vs the previous month "
                                  f"({fmt_money(d['current_30d'], lang_cur, 'en')} in the last 30 days).",
                            "fa": f"هزینهٔ {cat_lbl(d['category'],'fa')} نسبت به ماه قبل "
                                  f"{abs(d['delta_pct']):.0f}٪ {'افزایش' if up else 'کاهش'} یافته است "
                                  f"({fmt_money(d['current_30d'], lang_cur, 'fa')} در ۳۰ روز اخیر)."}})
        for a in self.anomalies(user_id, profile):
            if a["kind"] == "category_spike":
                out.append({"insight_id": "in_" + uuid.uuid4().hex[:6], "type": "anomaly",
                            "severity": "warning", "category": a["category"], "facts": a,
                            "message": {
                                "en": f"Unusual week for {cat_lbl(a['category'],'en')}: "
                                      f"{fmt_money(a['this_week'], lang_cur, 'en')} vs a typical "
                                      f"{fmt_money(a['baseline_week'], lang_cur, 'en')} (z={a['z']}).",
                                "fa": f"هفتهٔ غیرعادی برای {cat_lbl(a['category'],'fa')}: "
                                      f"{fmt_money(a['this_week'], lang_cur, 'fa')} در برابر میانگین "
                                      f"{fmt_money(a['baseline_week'], lang_cur, 'fa')}."},
                            "evidence_tx_ids": a["tx_ids"][:20]})
            else:
                out.append({"insight_id": "in_" + uuid.uuid4().hex[:6], "type": "anomaly",
                            "severity": "info", "category": a["category"], "facts": a,
                            "message": {
                                "en": f"Large purchase at {a['payee'] or 'unknown payee'}: "
                                      f"{fmt_money(a['amount'], lang_cur, 'en')} — well above your usual size.",
                                "fa": f"خرید بزرگ از {a['payee'] or 'پذیرندهٔ نامشخص'}: "
                                      f"{fmt_money(a['amount'], lang_cur, 'fa')} — بسیار بالاتر از حد معمول شما."},
                            "evidence_tx_ids": a["tx_ids"]})
        proj = self.projection(user_id, profile)
        if proj["budget"] and proj["overrun_probability"] >= 0.4 and proj["days_elapsed"] >= 5:
            out.append({"insight_id": "in_" + uuid.uuid4().hex[:6], "type": "budget",
                        "severity": "critical" if proj["overrun_probability"] >= 0.7 else "warning",
                        "category": None,
                        "facts": {k: v for k, v in proj.items() if k != "bootstrap_samples"},
                        "message": {
                            "en": f"At the current pace you'll end the month at "
                                  f"{fmt_money(proj['projected_month_end'], lang_cur, 'en')} — "
                                  f"{proj['overrun_probability']*100:.0f}% chance of exceeding your "
                                  f"{fmt_money(proj['budget'], lang_cur, 'en')} budget.",
                            "fa": f"با روند فعلی، هزینهٔ پایان ماه حدود "
                                  f"{fmt_money(proj['projected_month_end'], lang_cur, 'fa')} خواهد بود — "
                                  f"احتمال عبور از بودجهٔ {fmt_money(proj['budget'], lang_cur, 'fa')} "
                                  f"برابر {proj['overrun_probability']*100:.0f}٪ است."}})
        return out


analysis = AnalysisEngine()
print("analysis engine ready — projection, deltas, anomalies, bilingual insight facts")

analysis engine ready — projection, deltas, anomalies, bilingual insight facts


In [6]:
ELASTICITY = {"food_dining": "elastic", "entertainment": "elastic", "shopping": "elastic",
              "subscriptions": "elastic", "groceries": "semi", "transport": "semi",
              "education": "semi", "health": "inelastic", "housing_utilities": "inelastic",
              "transfers": "semi", "other": "semi"}
NEEDS = {"groceries", "transport", "housing_utilities", "health", "education"}
WANTS = {"food_dining", "entertainment", "shopping", "subscriptions", "other"}


class BudgetAdvisor:

    def status(self, user_id, profile):
        proj = analysis.projection(user_id, profile)
        burn = proj["daily_burn_28d"]
        balance = profile["balance"]["amount"]
        runway = round(balance / burn, 1) if burn > 0 else None
        risk = ("high" if proj["overrun_probability"] >= 0.6 else
                "medium" if proj["overrun_probability"] >= 0.25 else "low") if proj["budget"] else "unset"
        return {"balance": dict(profile["balance"]),
                "monthly_budget": dict(profile["monthly_budget"]),
                "projection": {k: v for k, v in proj.items() if k != "bootstrap_samples"},
                "runway_days": runway, "overrun_risk": risk,
                "low_balance": bool(runway is not None and runway < 7)}

    def allocation(self, user_id, profile):
        """50/30/20 needs/wants/savings framing against actual behavior."""
        mtd = analysis.month_to_date(user_id, profile)
        spent = mtd["spent"] or 1.0
        needs = sum(v for c, v in mtd["by_category"].items() if c in NEEDS)
        wants = sum(v for c, v in mtd["by_category"].items() if c in WANTS)
        income_base = mtd["income"] or profile["monthly_budget"]["amount"] or spent
        saved = max(income_base - spent, 0)
        return {"income_base": round(income_base, 2),
                "actual": {"needs_pct": round(needs / income_base * 100, 1) if income_base else 0,
                           "wants_pct": round(wants / income_base * 100, 1) if income_base else 0,
                           "savings_pct": round(saved / income_base * 100, 1) if income_base else 0},
                "target": {"needs_pct": 50, "wants_pct": 30, "savings_pct": 20},
                "currency": profile["currency"]}

    def savings_opportunities(self, user_id, profile):
        out, lang, cur = [], profile["language"], profile["currency"]
        for d in analysis.category_deltas(user_id, profile):
            c = d["category"]
            if ELASTICITY.get(c) != "elastic" or d["delta_pct"] <= 10 or d["current_30d"] <= 0:
                continue
            excess = d["current_30d"] - d["previous_30d"]
            save = round(max(excess * 0.7, d["current_30d"] * 0.15), 2)
            lbl = CATEGORY_LABELS[c]
            out.append({"category": c, "monthly_saving_estimate": save, "currency": cur,
                        "facts": d,
                        "message": {
                            "en": f"Bringing {lbl['en']} back to last month's level would free "
                                  f"≈ {fmt_money(save, cur, 'en')}/month.",
                            "fa": f"بازگرداندن {lbl['fa']} به سطح ماه قبل حدود "
                                  f"{fmt_money(save, cur, 'fa')} در ماه آزاد می‌کند."}})
        return sorted(out, key=lambda r: -r["monthly_saving_estimate"])[:5]


class HealthScore:
    """0–100, four explainable components. Judges can drill into every number."""

    WEIGHTS = {"budget_discipline": 0.35, "savings_rate": 0.25,
               "spending_stability": 0.20, "category_balance": 0.20}

    def compute(self, user_id, profile):
        proj = analysis.projection(user_id, profile)
        mtd = analysis.month_to_date(user_id, profile)

        if proj["budget"] > 0:
            ratio = proj["projected_month_end"] / proj["budget"]
            budget_score = float(np.clip((1.6 - ratio) / 0.6, 0, 1)) * 100
        else:
            budget_score = 50.0
        income_base = mtd["income"] or proj["budget"] or 0
        savings_score = float(np.clip((income_base - mtd["spent"]) / income_base / 0.25, 0, 1)) * 100 \
            if income_base > 0 else 50.0
        daily = [v["total"] for v in analysis.daily_series(user_id, profile, 28).values()]
        mu = float(np.mean(daily)) if daily else 0.0
        cv = (float(np.std(daily)) / mu) if mu > 0 else 0.0
        stability_score = float(np.clip(1 - (cv - 0.4) / 1.6, 0, 1)) * 100
        shares = np.array([v for v in mtd["by_category"].values() if v > 0], dtype=float)
        if len(shares) >= 2:
            p = shares / shares.sum()
            balance_score = float(-(p * np.log(p)).sum() / np.log(len(p))) * 100
        else:
            balance_score = 40.0 if len(shares) == 1 else 50.0
        comps = {"budget_discipline": round(budget_score, 1), "savings_rate": round(savings_score, 1),
                 "spending_stability": round(stability_score, 1), "category_balance": round(balance_score, 1)}
        total = round(sum(comps[k] * w for k, w in self.WEIGHTS.items()), 1)
        grade = ("excellent" if total >= 80 else "good" if total >= 60 else
                 "fair" if total >= 40 else "needs_attention")
        grade_label = {"excellent": {"en": "Excellent", "fa": "عالی"},
                       "good": {"en": "Good", "fa": "خوب"},
                       "fair": {"en": "Fair", "fa": "متوسط"},
                       "needs_attention": {"en": "Needs attention", "fa": "نیازمند توجه"}}[grade]
        return {"score": total, "grade": grade, "grade_label": grade_label,
                "components": comps, "weights": self.WEIGHTS}


class WhatIfSimulator:
    """'What if I cut restaurant spending 30%?' — live re-projection, reusing the same math."""

    def simulate(self, user_id, profile, adjustments: dict):
        adjustments = {c: float(np.clip(p, -100, 100)) for c, p in (adjustments or {}).items()
                       if c in CATEGORIES}
        proj = analysis.projection(user_id, profile)
        mtd = analysis.month_to_date(user_id, profile)
        hist = analysis.daily_series(user_id, profile, 28)
        n_days = len(hist) or 1
        cat_daily = defaultdict(float)
        for v in hist.values():
            for c, amt in v["by_category"].items():
                cat_daily[c] += amt / n_days
        remaining_days = proj["days_in_month"] - proj["days_elapsed"]
        base_rest = sum(cat_daily.values()) * remaining_days
        adj_rest = sum(rate * (1 + adjustments.get(c, 0) / 100) for c, rate in cat_daily.items()) \
            * remaining_days
        new_projection = round(mtd["spent"] + adj_rest, 2)
        return {"baseline_projection": proj["projected_month_end"],
                "adjusted_projection": new_projection,
                "monthly_saving": round(max(base_rest - adj_rest, 0), 2),
                "budget": proj["budget"],
                "now_within_budget": bool(proj["budget"] and new_projection <= proj["budget"]),
                "adjustments": adjustments, "currency": profile["currency"]}


class SubscriptionDetector:
    """Recurring payments from interval regularity on (normalized payee, ~amount)."""

    def detect(self, user_id, profile):
        txs = store.list(user_id, profile,
                         start=(user_now(profile).date() - timedelta(days=120)).isoformat(),
                         limit=1000)
        groups = defaultdict(list)
        for t in txs:
            if t["type"] == "income" or not t["payee"]:
                continue
            amt = t["total"]["amount"]
            groups[(normalize_text(t["payee"]), round(math.log10(max(amt, 1)), 1))].append(t)
        found = []
        for (payee_key, _), ts in groups.items():
            if len(ts) < 3:
                continue
            days = sorted(datetime.fromisoformat(t["timestamp"]).date() for t in ts)
            gaps = [(b - a).days for a, b in zip(days, days[1:]) if (b - a).days > 0]
            if not gaps:
                continue
            mean_gap, sd_gap = float(np.mean(gaps)), float(np.std(gaps))
            cadence = ("monthly" if 25 <= mean_gap <= 35 else
                       "weekly" if 6 <= mean_gap <= 8 else None)
            if cadence and sd_gap <= 5:
                amts = [t["total"]["amount"] for t in ts]
                found.append({"payee": ts[0]["payee"], "cadence": cadence,
                              "typical_amount": round(float(np.median(amts)), 2),
                              "currency": ts[0]["currency"],
                              "occurrences": len(ts),
                              "monthly_cost": round(float(np.median(amts)) *
                                                    (1 if cadence == "monthly" else 4.33), 2),
                              "next_expected": (days[-1] + timedelta(days=round(mean_gap))).isoformat(),
                              "category": ts[0]["category"]})
        return sorted(found, key=lambda r: -r["monthly_cost"])


advisor = BudgetAdvisor()
health = HealthScore()
whatif = WhatIfSimulator()
subscriptions = SubscriptionDetector()
print("advisor stack ready — budget status, health score, what-if simulator, subscription detector")

advisor stack ready — budget status, health score, what-if simulator, subscription detector


## 5 · OCR pipeline — Persian + English receipts

**Architecture: OCR for perception, LLM for understanding, regex as the deterministic floor.**

1. **Preprocess** (PIL, + CLAHE via OpenCV when available) — EXIF orientation fix, resize to ≤1600px,
   grayscale, autocontrast. Thermal receipts gain more from this than from model choice.
2. **EasyOCR `['fa','en']`** — the only mainstream open OCR that reads Persian and English in one
   pass. It is **lazy-loaded on first use** (the weights are ~1 GB) and entirely optional: without it
   the endpoint returns a clean bilingual `ocr_unavailable` error and the rest of the app is unaffected.
3. **Normalize** — Persian/Arabic-Indic digits → ASCII, ي/ك → ی/ک, currency-token detection
   (ریال / تومان / $), and a **Rial-magnitude guard**: if the receipt says ریال, amounts are ÷10 to
   Toman — the single most dangerous bilingual bug (silent 10× errors), handled explicitly.
4. **Extract** — the raw text goes to the LLM with a strict JSON-only schema (temperature 0, "use
   null, never guess prices"), then is validated: line totals must reconcile with the receipt total
   within 2% or the draft is flagged `needs_review`. **Fallback:** a deterministic per-line
   price-pattern extractor when no LLM key is configured.
5. **Human-in-the-loop always** — the endpoint returns a *draft*; nothing is saved until the user
   confirms in the review table. OCR imperfection becomes UX, not corruption.

In [7]:
class OCRService:

    def __init__(self):
        self._reader = None
        self._reader_lock = threading.Lock()

    @property
    def available(self):
        return CAPS["easyocr"] and CAPS["pillow"]

    def _get_reader(self):
        with self._reader_lock:
            if self._reader is None:
                import easyocr
                self._reader = easyocr.Reader(["fa", "en"], gpu=False, verbose=False)
            return self._reader

    def preprocess(self, image_bytes: bytes) -> np.ndarray:
        img = Image.open(io.BytesIO(image_bytes))
        img = ImageOps.exif_transpose(img)
        if max(img.size) > 1600:
            img.thumbnail((1600, 1600), Image.LANCZOS)
        img = ImageOps.autocontrast(img.convert("L"), cutoff=1)
        arr = np.array(img)
        try:
            import cv2
            arr = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(arr)
            arr = cv2.fastNlMeansDenoising(arr, h=8)
        except ImportError:
            pass
        return arr

    def read_text(self, image_bytes: bytes):
        arr = self.preprocess(image_bytes)
        results = self._get_reader().readtext(arr, detail=1, paragraph=False)
        boxes = []
        for box, text, conf in results:
            ys = [p[1] for p in box]
            xs = [p[0] for p in box]
            boxes.append({"y": sum(ys) / 4, "x": min(xs), "text": text, "conf": float(conf)})
        boxes.sort(key=lambda b: b["y"])
        lines, current, last_y = [], [], None
        for b in boxes:
            if last_y is not None and b["y"] - last_y > 14:
                lines.append(sorted(current, key=lambda t: t["x"]))
                current = []
            current.append(b)
            last_y = b["y"]
        if current:
            lines.append(sorted(current, key=lambda t: t["x"]))
        text_lines = ["  ".join(t["text"] for t in ln) for ln in lines]
        mean_conf = float(np.mean([b["conf"] for b in boxes])) if boxes else 0.0
        return normalize_text_lines(text_lines), mean_conf

    def regex_extract(self, lines, default_currency):
        price_re = re.compile(r"(\d[\d,\.]{2,})\s*$")
        items, total = [], None
        total_words = ("جمع", "مبلغ کل", "قابل پرداخت", "total", "amount due", "sum")
        for ln in lines:
            m = price_re.search(ln.replace("٬", ",").strip())
            if not m:
                continue
            try:
                val = float(m.group(1).replace(",", ""))
            except ValueError:
                continue
            name = ln[: m.start()].strip(" -.:*").strip()
            if any(w in ln.lower() for w in total_words):
                total = max(total or 0, val)
            elif name and val > 0:
                items.append({"name": name[:120], "quantity": 1, "unit_price": val, "line_total": val})
        if total is None and items:
            total = sum(i["line_total"] for i in items)
        return {"merchant": None, "items": items, "total": total,
                "currency": default_currency, "method": "regex"}

    def process(self, user_id, profile, image_bytes: bytes, filename: str):
        lines, conf = self.read_text(image_bytes)
        raw_text = "\n".join(lines)
        detected_rial = "ریال" in raw_text
        default_currency = profile["currency"]

        extracted = None
        if llm.available and raw_text.strip():
            extracted = llm.extract_receipt(raw_text, default_currency)
        if not extracted or not isinstance(extracted.get("items"), list):
            extracted = self.regex_extract(lines, default_currency)

        currency = extracted.get("currency") or default_currency
        scale = 0.1 if (detected_rial and currency == "TOMAN") else 1.0
        items, items_sum = [], 0.0
        for it in extracted.get("items", [])[:60]:
            try:
                unit = float(it.get("unit_price") or it.get("line_total") or 0) * scale
                qty = max(1, int(it.get("quantity") or 1))
                name = str(it.get("name", "")).strip()
            except (TypeError, ValueError):
                continue
            if name and unit > 0:
                line = round(unit * qty, 2)
                items.append({"name": name[:120], "quantity": qty, "unit_price": unit,
                              "line_total": line})
                items_sum += line
        total = float(extracted.get("total") or 0) * scale
        if total <= 0:
            total = round(items_sum, 2)
        reconciles = bool(total > 0 and items and abs(items_sum - total) / total <= 0.02)

        rc_name = f"rc_{datetime.now(timezone.utc):%Y%m%d%H%M%S}_{uuid.uuid4().hex[:6]}" \
                  f"{os.path.splitext(filename or '')[1] or '.jpg'}"
        os.makedirs(receipts_dir(user_id), exist_ok=True)
        with open(os.path.join(receipts_dir(user_id), rc_name), "wb") as fh:
            fh.write(image_bytes)

        return {"draft_transaction": {
                    "type": "purchase", "source": "ocr_receipt",
                    "payee": (extracted.get("merchant") or "")[:120] if extracted.get("merchant") else "",
                    "amount": total, "currency": currency, "items": items,
                    "ocr": {"receipt_image": f"receipts/{rc_name}",
                            "raw_text_excerpt": raw_text[:800], "confidence": round(conf, 2)}},
                "raw_text": raw_text, "ocr_confidence": round(conf, 2),
                "extraction_method": extracted.get("method", "llm"),
                "rial_converted": scale != 1.0,
                "needs_review": bool(not reconciles or conf < 0.55)}


def normalize_text_lines(lines):
    return [normalize_text_keepcase(ln) for ln in lines if ln.strip()]


def normalize_text_keepcase(s: str) -> str:
    return (s or "").translate(_FA_TRANS).strip()


ocr = OCRService()
print(f"OCR service ready — engine {'available' if ocr.available else 'NOT installed (lazy/optional): pip install easyocr'}")

OCR service ready — engine available


## 6 · Bitcoin service — serving the `bitcoin.ipynb` artifacts

Loads `data/btc/model.keras` + `scalers.pkl` + `metadata.json` **once** at startup and serves
millisecond forecasts. Feature computation is byte-identical to the training notebook (same 20
features, same 250-day context the training notebook verified to `1e-10` against full history).

**Price source chain (never breaks a demo):**
1. **CryptoCompare** `histoday` — true daily OHLCV, no API key;
2. **CoinGecko** `market_chart` — daily closes + volume; OHLC is approximated from adjacent closes
   (documented approximation, return-space features tolerate it);
3. **Cached** last successful fetch (`data/btc/price_cache.json`);
4. **Dataset fallback** — tail of `datasets/btc/btc_test.csv`, so the module works fully offline.

Every response carries its `source` so the UI can label the data honestly. Missing artifacts (if
`bitcoin.ipynb` hasn't been run) degrade to a clean bilingual `btc_unavailable` error — nothing else
in the app is affected.

In [8]:
def btc_compute_features(ohlcv: pd.DataFrame) -> pd.DataFrame:
    """EXACT replica of compute_features() in bitcoin.ipynb — the serving contract."""
    c, h, l, v = ohlcv["Close"], ohlcv["High"], ohlcv["Low"], ohlcv["Volume"]
    logc = np.log(c)
    r = logc.diff()
    feat = pd.DataFrame(index=ohlcv.index)
    feat["log_ret"] = r
    rng = h - l
    feat["hl_range"] = rng / c
    feat["close_pos"] = ((c - l) / rng.replace(0, np.nan)).fillna(0.5)
    logv = np.log(v)
    feat["log_vol_chg"] = logv.diff()
    feat["vol_ratio"] = np.log(v / v.rolling(20).mean())
    for w in (7, 14, 30):
        feat[f"volat_{w}"] = r.rolling(w).std()
    delta = c.diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    rs = gain / loss.replace(0, np.nan)
    feat["rsi_14"] = ((100 - 100 / (1 + rs)) / 100.0).fillna(0.5)
    ema12 = logc.ewm(span=12, adjust=False).mean()
    ema26 = logc.ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    sig = macd.ewm(span=9, adjust=False).mean()
    feat["macd"], feat["macd_signal"], feat["macd_hist"] = macd, sig, macd - sig
    m20, s20 = c.rolling(20).mean(), c.rolling(20).std()
    feat["bb_pctb"] = ((c - (m20 - 2 * s20)) / (4 * s20).replace(0, np.nan)).clip(-0.5, 1.5)
    for w in (7, 21, 50):
        feat[f"close_sma{w}"] = np.log(c / c.rolling(w).mean())
    feat["roc_7"] = logc.diff(7)
    feat["roc_14"] = logc.diff(14)
    dow = ohlcv.index.dayofweek
    feat["dow_sin"] = np.sin(2 * np.pi * dow / 7)
    feat["dow_cos"] = np.cos(2 * np.pi * dow / 7)
    return feat


class BTCService:
    CACHE_PATH = os.path.join(CONFIG["btc_artifacts_dir"], "price_cache.json")

    def __init__(self):
        self.model = self.scalers = self.metadata = None
        self._lock = threading.Lock()
        self._load_lock = threading.Lock()
        self._last_refresh = 0.0
        self._last_load_attempt = 0.0
        self._try_load()

    def _try_load(self):
        """Best-effort artifact load. Retried lazily (throttled) so a model trained
        AFTER the server started is picked up on the next request — no restart needed."""
        if not CAPS["tensorflow"] or self.model is not None:
            return
        with self._load_lock:
            if self.model is not None or time.time() - self._last_load_attempt < 30:
                return
            self._last_load_attempt = time.time()
            try:
                model = keras.models.load_model(
                    os.path.join(CONFIG["btc_artifacts_dir"], "model.keras"))
                with open(os.path.join(CONFIG["btc_artifacts_dir"], "scalers.pkl"), "rb") as fh:
                    scalers = pickle.load(fh)
                self.metadata = read_json(os.path.join(CONFIG["btc_artifacts_dir"], "metadata.json"))
                self.model, self.scalers = model, scalers
                print(f"[btc] model artifacts loaded from {CONFIG['btc_artifacts_dir']}")
            except Exception:
                self.model = self.scalers = None

    @property
    def available(self):
        self._try_load()
        return self.model is not None and self.scalers is not None

    def _fetch_cryptocompare(self):
        r = requests.get("https://min-api.cryptocompare.com/data/v2/histoday",
                         params={"fsym": "BTC", "tsym": "USD", "limit": 260}, timeout=8)
        r.raise_for_status()
        rows = [{"date": datetime.fromtimestamp(d["time"], tz=timezone.utc).date().isoformat(),
                 "Open": d["open"], "High": d["high"], "Low": d["low"], "Close": d["close"],
                 "Volume": max(d["volumeto"], 1.0)}
                for d in r.json()["Data"]["Data"] if d.get("close")]
        return rows, "cryptocompare"

    def _fetch_coingecko(self):
        r = requests.get("https://api.coingecko.com/api/v3/coins/bitcoin/market_chart",
                         params={"vs_currency": "usd", "days": 260, "interval": "daily"}, timeout=8)
        r.raise_for_status()
        js = r.json()
        prices = {datetime.fromtimestamp(t / 1000, tz=timezone.utc).date().isoformat(): p
                  for t, p in js["prices"]}
        vols = {datetime.fromtimestamp(t / 1000, tz=timezone.utc).date().isoformat(): v
                for t, v in js["total_volumes"]}
        rows, prev = [], None
        for d in sorted(prices):
            close = prices[d]
            opn = prev if prev is not None else close
            rows.append({"date": d, "Open": opn, "High": max(opn, close),
                         "Low": min(opn, close), "Close": close,
                         "Volume": max(vols.get(d, 1.0), 1.0)})
            prev = close
        return rows, "coingecko_approx"

    def _dataset_fallback(self):
        raw = pd.read_csv(CONFIG["btc_dataset_fallback"], encoding="utf-8-sig")
        raw["date"] = pd.to_datetime(raw["Start"].astype(str).str.slice(0, 10)).dt.date.astype(str)
        raw = raw.sort_values("date").tail(260)
        return [{"date": r["date"], "Open": r["Open"], "High": r["High"], "Low": r["Low"],
                 "Close": r["Close"], "Volume": max(float(r["Volume"]), 1.0)}
                for _, r in raw.iterrows()], "dataset_fallback"

    def get_rows(self, force_refresh=False):
        with self._lock:
            cache = read_json(self.CACHE_PATH)
            fresh = cache and (time.time() - self._last_refresh) < CONFIG["btc_refresh_s"]
            if fresh and not force_refresh:
                return cache["rows"], cache["source"]
            for fetch in (self._fetch_cryptocompare, self._fetch_coingecko):
                try:
                    rows, source = fetch()
                    if len(rows) >= 80:
                        write_json_atomic(self.CACHE_PATH, {
                            "source": source, "rows": rows,
                            "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
                        self._last_refresh = time.time()
                        return rows, source
                except (requests.RequestException, KeyError, ValueError):
                    continue
            if cache:
                return cache["rows"], cache["source"] + "_stale"
            rows, source = self._dataset_fallback()
            write_json_atomic(self.CACHE_PATH, {
                "source": source, "rows": rows,
                "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
            return rows, source

    def history(self, days=30):
        rows, source = self.get_rows()
        tail = rows[-days:]
        return {"days": [{"date": r["date"], "close": round(r["Close"], 2)} for r in tail],
                "latest_price": round(tail[-1]["Close"], 2), "source": source}

    def forecast(self):
        if not self.available:
            return None
        rows, source = self.get_rows()
        frame = pd.DataFrame(rows).set_index(pd.to_datetime([r["date"] for r in rows]))
        frame = frame[["Open", "High", "Low", "Close", "Volume"]].astype(float)
        feat = btc_compute_features(frame).dropna()
        lookback = self.scalers["lookback"]
        if len(feat) < lookback:
            return None
        window = self.scalers["feature_scaler"].transform(
            feat[self.scalers["feature_names"]].tail(lookback).to_numpy()).astype(np.float32)[None, ...]
        pred_ret = self.scalers["target_scaler"].inverse_transform(
            self.model.predict(window, verbose=0))[0]
        last_date = feat.index[-1].date()
        last_close = float(frame["Close"].iloc[-1])
        cum = np.cumsum(pred_ret)
        prices = last_close * np.exp(cum)
        sigcal = (self.metadata or {}).get("signal_calibration", {})
        thr = float(sigcal.get("move_threshold_log_return", 0.015))
        err_std = float(sigcal.get("cum10_error_std_log_return", 0.10))
        total = float(cum[-1])
        signal = "UP" if total > thr else "DOWN" if total < -thr else "HOLD"
        return {"as_of": last_date.isoformat(), "last_close": round(last_close, 2),
                "source": source,
                "forecast": [{"date": (last_date + timedelta(days=k + 1)).isoformat(),
                              "price": round(float(prices[k]), 2),
                              "cum_move_pct": round(float(np.expm1(cum[k])) * 100, 2)}
                             for k in range(len(pred_ret))],
                "signal": {"signal": signal,
                           "confidence": round(float(min(abs(total) / (2 * thr), 1.0)), 3),
                           "expected_move_pct": round(float(np.expm1(total)) * 100, 2),
                           "uncertainty_pct": round(float(np.expm1(err_std)) * 100, 2)},
                "model_metrics": {
                    "validation": (self.metadata or {}).get("validation_metrics", {}).get("vs_baselines"),
                    "final_test_acting_hit_rate_pct": sigcal.get("final_test_acting_hit_rate_pct"),
                },
                "caveats": (self.metadata or {}).get("caveats", [])}


btc = BTCService()
print(f"BTC service ready — model {'LOADED' if btc.available else 'not found (run bitcoin.ipynb)'} · "
      "price chain: cryptocompare → coingecko → cache → dataset")

[btc] model artifacts loaded from data/btc
BTC service ready — model LOADED · price chain: cryptocompare → coingecko → cache → dataset


In [9]:
import calendar as _calendar

FINANCE_ART_DIR = "data/finance"


def finance_recent_months(user_id, profile, n_months):
    """Last n_months of monthly aggregates from the user's transaction JSON, oldest→newest.
    Currency-consistent (account currency only); crypto fields are 0 (not tracked per user)."""
    cur = profile["currency"]
    today = user_now(profile).date()
    y, m = today.year, today.month
    spans = []
    for k in range(n_months - 1, -1, -1):
        mm, yy = m - k, y
        while mm <= 0:
            mm += 12; yy -= 1
        first = date(yy, mm, 1)
        last = date(yy, mm, _calendar.monthrange(yy, mm)[1])
        if last > today:
            last = today
        spans.append((yy, mm, first, last))
    out = []
    for (yy, mm, first, last) in spans:
        spent = income = weekend = largest = 0.0
        by_cat, n = defaultdict(float), 0
        for doc in store.days_in_range(user_id, first, last):
            for t in doc["transactions"]:
                if t["currency"] != cur:
                    continue
                amt = t["total"]["amount"]
                if t["type"] == "income":
                    income += amt
                    continue
                spent += amt
                by_cat[t["category"]] += amt
                n += 1
                largest = max(largest, amt)
                try:
                    if datetime.fromisoformat(t["timestamp"]).weekday() >= 5:
                        weekend += amt
                except (ValueError, KeyError):
                    pass
        budget = profile["monthly_budget"]["amount"] or 0
        row = {"spend": spent, "income": income if income > 0 else max(spent, 1.0),
               "budget_util": (spent / budget) if budget > 0 else 1.0,
               "savings_rate": ((income - spent) / income) if income > 0 else 0.0,
               "n_transactions": n, "weekend_ratio": (weekend / spent) if spent > 0 else 0.3,
               "largest_txn": largest, "crypto_value": 0.0, "crypto_txn": 0,
               "month_of_year": mm}
        for c in CATEGORIES:
            if c != "income":
                row[f"cat_{c}"] = by_cat.get(c, 0.0)
        out.append(row)
    return out


class FinanceModelService:

    def __init__(self):
        self.model = self.scaler = self.meta = None
        self._lock = threading.Lock()
        self._last_attempt = 0.0
        self._try_load()

    def _try_load(self):
        if not CAPS["tensorflow"] or self.model is not None:
            return
        with self._lock:
            if self.model is not None or time.time() - self._last_attempt < 30:
                return
            self._last_attempt = time.time()
            try:
                self.model = keras.models.load_model(os.path.join(FINANCE_ART_DIR, "finance_model.keras"))
                with open(os.path.join(FINANCE_ART_DIR, "finance_scaler.pkl"), "rb") as fh:
                    self.scaler = pickle.load(fh)
                self.meta = read_json(os.path.join(FINANCE_ART_DIR, "finance_meta.json"))
                print("[finance] behaviour model loaded")
            except Exception:
                self.model = self.scaler = None

    @property
    def available(self):
        self._try_load()
        return self.model is not None and self.scaler is not None

    def _row_features(self, cur, prev, cats):
        """Mirror of month_feature_row() in Financa.ipynb — order MUST match feature_names."""
        s = float(cur["spend"]); inc = float(cur["income"]); ps = float(prev["spend"])
        row = [
            float(np.clip(s / (inc + 1e-6), 0, 3)),
            float(np.clip(cur["budget_util"], 0, 3)),
            float(np.clip(cur["savings_rate"], -2, 1)),
            float(np.clip(np.log((s + 1e-6) / (ps + 1e-6)), -1.5, 1.5)),
            float(cur["weekend_ratio"]),
            float(np.clip(cur["largest_txn"] / (s + 1e-6), 0, 1)),
            float(np.log1p(cur["n_transactions"])),
            float(np.tanh(cur["crypto_value"] / (s + 1e-6) / 2)),
            float(np.clip(cur["crypto_txn"] / 10.0, 0, 2)),
            float(np.sin(2 * np.pi * cur["month_of_year"] / 12)),
            float(np.cos(2 * np.pi * cur["month_of_year"] / 12)),
        ]
        for c in cats:
            row.append(float(np.clip(cur[f"cat_{c}"] / (s + 1e-6), 0, 1)))
        return row

    def predict(self, user_id, profile):
        if not self.available:
            return {"available": False}
        sc, meta = self.scaler, self.meta or {}
        L, cats = sc["lookback"], sc["spend_cats"]
        months = finance_recent_months(user_id, profile, L + 1)

        while len(months) < L + 1:
            months.insert(0, dict(months[0]))
        feats = [self._row_features(months[i], months[i - 1], cats) for i in range(1, L + 1)]
        X = np.asarray(feats, dtype=np.float32)
        Xs = sc["feature_scaler"].transform(X).reshape(1, L, len(cats) + 11).astype(np.float32)
        reg_s, prob = self.model.predict(Xs, verbose=0)
        growth = float(reg_s.ravel()[0]) * sc["reg_std"] + sc["reg_mean"]
        prob = float(prob.ravel()[0])
        current_spend = float(months[-1]["spend"])
        predicted = round(current_spend * float(np.exp(growth)), 2)
        risk = "high" if prob >= 0.6 else "medium" if prob >= 0.35 else "low"
        
        cur = months[-1]
        shares = {c: cur[f"cat_{c}"] for c in cats if cur["spend"] > 0}
        top_cat = max(shares, key=shares.get) if shares else None
        lang_cur = profile["currency"]
        tips = self._tips(risk, growth, top_cat, predicted, current_spend, lang_cur)
        return {"available": True,
                "predicted_next_spend": {"amount": predicted, "currency": lang_cur},
                "current_spend": {"amount": round(current_spend, 2), "currency": lang_cur},
                "growth_pct": round((np.exp(growth) - 1) * 100, 1),
                "overspend_probability": round(prob, 3), "risk": risk,
                "top_category": top_cat,
                "tips": tips,
                "model_metrics": meta.get("metrics", {})}

    def _tips(self, risk, growth, top_cat, predicted, current, cur):
        up = growth > 0.02
        cl = CATEGORY_LABELS.get(top_cat, CATEGORY_LABELS["other"]) if top_cat else None
        en, fa = [], []
        if risk == "high":
            en.append("High overspend risk next month — set aside your fixed bills first and cap discretionary categories.")
            fa.append("خطر بالای عبور از بودجه در ماه آینده — ابتدا قبوض ثابت را کنار بگذارید و هزینه‌های اختیاری را محدود کنید.")
        elif risk == "medium":
            en.append("Moderate overspend risk — a light budget review this week keeps you on track.")
            fa.append("خطر متوسط عبور از بودجه — یک بازبینی کوتاه بودجه در این هفته شما را در مسیر نگه می‌دارد.")
        else:
            en.append("Low overspend risk — your recent pattern is on track. Consider moving the surplus to savings.")
            fa.append("خطر پایین عبور از بودجه — روند اخیر شما مناسب است. مازاد را به پس‌انداز منتقل کنید.")
        if up:
            en.append(f"Spending is trending up (~{(np.exp(growth)-1)*100:.0f}% vs this month)"
                      + (f", led by {cl['en']}." if cl else "."))
            fa.append(f"روند هزینه صعودی است (حدود {(np.exp(growth)-1)*100:.0f}٪ نسبت به این ماه)"
                      + (f"، بیشتر در دستهٔ {cl['fa']}." if cl else "."))
        else:
            en.append("Spending is flat-to-down next month — a good window to lock in savings.")
            fa.append("هزینهٔ ماه آینده ثابت یا کاهشی است — فرصت خوبی برای تثبیت پس‌انداز.")
        return {"en": en, "fa": fa}


finance = FinanceModelService()
print(f"finance behaviour model: {'LOADED' if finance.available else 'not found (run Financa.ipynb)'}")

[finance] behaviour model loaded
finance behaviour model: LOADED


/Users/sam/Desktop/aiolearn/venv/lib/python3.12/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 7 · LLM orchestrator — grounded chat, tools, streaming, `Searching…`

**Retrieval-then-answer, never context-dumping.** The model gets six tools over the user's real
data — `get_spending_summary`, `search_transactions`, `get_daily_files` (the literal last-30-days
JSON access required by the spec), `get_budget_status`, `get_insights`, `get_btc_context`. All
arithmetic happens in Python; the LLM narrates computed facts.

**Streaming contract** (NDJSON lines consumed by `AI.html`):
`{"type":"status","key":"searching"}` the moment a tool round begins → tools execute →
`{"type":"token","text":…}` for the final streamed answer → `{"type":"done"}`. This implements the
required *"Searching…"* UX honestly — the indicator shows exactly when file retrieval is happening.

**Resilience ladder:** full tool-calling → if the endpoint rejects the `tools` parameter, one retry
with pre-fetched context injected into the prompt → if no API key at all, a deterministic bilingual
answer built from the analysis engine. The chat never 500s because of a missing key.

Hallucination control: temperature 0.3, closed category enums, "answer money questions only from tool
results, say so when data is insufficient" as hard system rules, and BTC statements anchored to the
model signal object.

In [10]:
class LLMOrchestrator:

    def __init__(self):
        self.client = None
        self._tools_ok = True
        if CAPS["openai_sdk"] and GAPGPT_API_KEY:
            self.client = OpenAI(base_url=CONFIG["llm_base_url"], api_key=GAPGPT_API_KEY)

    @property
    def available(self):
        return self.client is not None

    @staticmethod
    def _parse_json(text):
        m = re.search(r"\{.*\}", text or "", re.DOTALL)
        try:
            return json.loads(m.group(0)) if m else None
        except json.JSONDecodeError:
            return None

    def _complete(self, messages, temperature=None, max_tokens=1200, **kw):
        return self.client.chat.completions.create(
            model=CONFIG["llm_model"], messages=messages,
            temperature=CONFIG["llm_temperature"] if temperature is None else temperature,
            max_tokens=max_tokens, **kw)

    def classify_category(self, text):
        try:
            resp = self._complete([{"role": "user", "content":
                "Classify this transaction into exactly one category id from this list: "
                + ", ".join(CATEGORIES) + ". Respond with the category id only, nothing else.\n\n"
                "Transaction (may be Persian or English): " + text[:400]}],
                temperature=0, max_tokens=10)
            return resp.choices[0].message.content.strip().lower()
        except Exception:
            return None

    def extract_receipt(self, raw_text, default_currency):
        try:
            resp = self._complete([{"role": "user", "content":
                "Extract structured data from this OCR'd receipt text (Persian and/or English).\n"
                "Respond with JSON ONLY, exactly this schema:\n"
                '{"merchant": string|null, "currency": "TOMAN"|"USD"|null,\n'
                ' "items": [{"name": string, "quantity": int, "unit_price": number, "line_total": number}],\n'
                ' "total": number|null}\n'
                "Rules: if a field is unreadable use null — NEVER guess prices. If the receipt shows "
                "Rial (ریال) amounts, still report the numbers as printed and set currency to TOMAN "
                "only if تومان appears. Default currency if unclear: " + default_currency +
                ".\n\nRECEIPT TEXT:\n" + raw_text[:4000]}],
                temperature=0, max_tokens=1500)
            parsed = self._parse_json(resp.choices[0].message.content)
            if parsed is not None:
                parsed["method"] = "llm"
            return parsed
        except Exception:
            return None

    def phrase_notification(self, ntype, facts, profile):
        """→ {'title': {en, fa}, 'body': {en, fa}} or None (caller falls back to templates)."""
        if not self.available:
            return None
        try:
            resp = self._complete([{"role": "user", "content":
                "Write a short financial app notification in BOTH English and Persian (Farsi).\n"
                f"Notification type: {ntype}. User country: {profile['country']}, "
                f"currency: {profile['currency']}.\n"
                "Use ONLY the numbers given in the facts — do not invent any figure. Warm, concise, "
                "actionable; no emoji; Persian must be natural, not machine-translated.\n"
                "Respond with JSON ONLY: "
                '{"title_en": str, "title_fa": str, "body_en": str, "body_fa": str}\n\n'
                "FACTS:\n" + json.dumps(facts, ensure_ascii=False, default=str)[:2000]}],
                temperature=0.5, max_tokens=400)
            p = self._parse_json(resp.choices[0].message.content)
            if p and all(k in p for k in ("title_en", "title_fa", "body_en", "body_fa")):
                return {"title": {"en": p["title_en"], "fa": p["title_fa"]},
                        "body": {"en": p["body_en"], "fa": p["body_fa"]}}
        except Exception:
            pass
        return None

    def _tool_specs(self):
        def spec(name, desc, props, req):
            return {"type": "function", "function": {"name": name, "description": desc,
                    "parameters": {"type": "object", "properties": props, "required": req}}}
        return [
            spec("get_spending_summary", "Aggregated spending for the last N days, total and per category.",
                 {"window_days": {"type": "integer", "description": "1-90"}}, ["window_days"]),
            spec("search_transactions", "Search the user's transactions.",
                 {"query": {"type": "string"}, "category": {"type": "string"},
                  "start_date": {"type": "string"}, "end_date": {"type": "string"},
                  "limit": {"type": "integer"}}, []),
            spec("get_daily_files", "Raw daily transaction JSON files for the last N days (max 30).",
                 {"last_n_days": {"type": "integer"}}, ["last_n_days"]),
            spec("get_budget_status", "Balance, budget, month-end projection, overrun risk, runway.", {}, []),
            spec("get_insights", "Current computed insights: trends, anomalies, budget alerts.", {}, []),
            spec("get_btc_context", "Latest Bitcoin price, 10-day model forecast and UP/DOWN/HOLD signal.", {}, []),
        ]

    def _run_tool(self, name, args, user_id, profile):
        try:
            if name == "get_spending_summary":
                days = int(np.clip(int(args.get("window_days", 30)), 1, 90))
                series = analysis.daily_series(user_id, profile, days)
                by_cat = defaultdict(float)
                for v in series.values():
                    for c, amt in v["by_category"].items():
                        by_cat[c] += amt
                return {"window_days": days, "currency": profile["currency"],
                        "total_spent": round(sum(v["total"] for v in series.values()), 2),
                        "by_category": {k: round(v, 2) for k, v in
                                        sorted(by_cat.items(), key=lambda kv: -kv[1])}}
            if name == "search_transactions":
                return {"transactions": store.list(
                    user_id, profile, start=args.get("start_date"), end=args.get("end_date"),
                    category=args.get("category"), query=args.get("query"),
                    limit=int(args.get("limit", 25)))[:25]}
            if name == "get_daily_files":
                n = int(np.clip(int(args.get("last_n_days", 30)), 1, 30))
                return {"daily_files": store.last_n_day_files(user_id, profile, n)}
            if name == "get_budget_status":
                return advisor.status(user_id, profile)
            if name == "get_insights":
                return {"insights": analysis.insights(user_id, profile)}
            if name == "get_btc_context":
                fc = btc.forecast()
                return fc if fc else {"error": "bitcoin model not available",
                                      "history": btc.history(30)}
            return {"error": f"unknown tool {name}"}
        except Exception as exc:
            return {"error": str(exc)[:300]}

    def _system_prompt(self, profile):
        today = user_now(profile).date().isoformat()
        return (
            "You are the AI Personal Finance Coach inside a bilingual (English/Persian) personal "
            "finance app for users in Iran and the USA.\n"
            f"User country: {profile['country']} · account currency: {profile['currency']} · "
            f"preferred language: {profile['language']}.\n"
            f"Today is {today} (Gregorian) / {jalali_of(today)} (Jalali).\n"
            "HARD RULES:\n"
            "1. Answer questions about the user's money ONLY from tool results. If you have not "
            "called a tool for a figure, call it. Never invent or estimate numbers.\n"
            "2. Always state amounts with their currency "
            f"({'Toman' if profile['currency'] == 'TOMAN' else 'USD'}).\n"
            "3. If the data is insufficient, say so plainly.\n"
            f"4. Respond in {'Persian (Farsi)' if profile['language'] == 'fa' else 'English'} unless "
            "the user writes in the other language.\n"
            "5. Bitcoin: only cite the model signal from get_btc_context, always with its confidence "
            "and uncertainty, and remind the user it is decision support, not financial advice.\n"
            "6. Be warm, specific and concise. Give actionable suggestions grounded in the numbers.")

    def chat_stream(self, user_id, profile, message, history):
        if not self.available:
            yield {"type": "status", "key": "searching"}
            yield from self._fallback_answer(user_id, profile)
            yield {"type": "done"}
            return
        messages = ([{"role": "system", "content": self._system_prompt(profile)}] +
                    [m for m in (history or [])[-10:]
                     if isinstance(m, dict) and m.get("role") in ("user", "assistant")
                     and isinstance(m.get("content"), str)] +
                    [{"role": "user", "content": str(message)[:4000]}])
        try:
            for _ in range(3):
                if not self._tools_ok:
                    break
                try:
                    resp = self._complete(messages, tools=self._tool_specs())
                except Exception:
                    self._tools_ok = False
                    break
                msg = resp.choices[0].message
                if not msg.tool_calls:
                    if msg.content:
                        yield {"type": "token", "text": msg.content}
                        yield {"type": "done"}
                        return
                    break
                yield {"type": "status", "key": "searching"}
                messages.append({"role": "assistant", "content": msg.content or "",
                                 "tool_calls": [{"id": tc.id, "type": "function",
                                                 "function": {"name": tc.function.name,
                                                              "arguments": tc.function.arguments}}
                                                for tc in msg.tool_calls]})
                for tc in msg.tool_calls:
                    try:
                        args = json.loads(tc.function.arguments or "{}")
                    except json.JSONDecodeError:
                        args = {}
                    result = self._run_tool(tc.function.name, args, user_id, profile)
                    messages.append({"role": "tool", "tool_call_id": tc.id,
                                     "content": json.dumps(result, ensure_ascii=False,
                                                           default=str)[:12000]})
            if not self._tools_ok:
                yield {"type": "status", "key": "searching"}
                ctx = {"spending": self._run_tool("get_spending_summary", {"window_days": 30},
                                                  user_id, profile),
                       "budget": self._run_tool("get_budget_status", {}, user_id, profile),
                       "btc": self._run_tool("get_btc_context", {}, user_id, profile)}
                messages = [m for m in messages if m["role"] in ("system", "user", "assistant")
                            and "tool_calls" not in m]
                messages.insert(1, {"role": "system", "content":
                                    "GROUND TRUTH DATA (answer only from this):\n"
                                    + json.dumps(ctx, ensure_ascii=False, default=str)[:12000]})
            stream = self._complete(messages, stream=True, max_tokens=1500)
            for chunk in stream:
                delta = chunk.choices[0].delta.content if chunk.choices else None
                if delta:
                    yield {"type": "token", "text": delta}
            yield {"type": "done"}
        except Exception as exc:
            yield {"type": "error", "code": "llm_unavailable", "detail": str(exc)[:200]}
            yield from self._fallback_answer(user_id, profile)
            yield {"type": "done"}

    def _fallback_answer(self, user_id, profile):
        """Deterministic grounded answer when no LLM is reachable — the app never goes silent."""
        lang, cur = profile["language"], profile["currency"]
        s = self._run_tool("get_spending_summary", {"window_days": 30}, user_id, profile)
        b = self._run_tool("get_budget_status", {}, user_id, profile)
        top = next(iter(s.get("by_category", {}).items()), None)
        if lang == "fa":
            text = ("دستیار هوشمند در حال حاضر در دسترس نیست (کلید API تنظیم نشده)، اما خلاصهٔ "
                    f"داده‌های شما: هزینهٔ ۳۰ روز اخیر {fmt_money(s.get('total_spent', 0), cur, 'fa')} است"
                    + (f"، بیشترین هزینه در دستهٔ {CATEGORY_LABELS[top[0]]['fa']} "
                       f"({fmt_money(top[1], cur, 'fa')})" if top else "")
                    + f". موجودی فعلی {fmt_money(b['balance']['amount'], cur, 'fa')} است.")
        else:
            text = ("The AI assistant is currently unavailable (API key not configured), but from "
                    f"your data: you spent {fmt_money(s.get('total_spent', 0), cur, 'en')} in the "
                    "last 30 days"
                    + (f", most of it on {CATEGORY_LABELS[top[0]]['en']} "
                       f"({fmt_money(top[1], cur, 'en')})" if top else "")
                    + f". Current balance: {fmt_money(b['balance']['amount'], cur, 'en')}.")
        yield {"type": "token", "text": text}

    def transcribe(self, audio_bytes, filename, language_hint):
        resp = self.client.audio.transcriptions.create(
            model=CONFIG["stt_model"],
            file=(filename or "audio.webm", audio_bytes),
            **({"language": language_hint} if language_hint in ("fa", "en") else {}))
        return resp.text


llm = LLMOrchestrator()
print(f"LLM orchestrator ready — {'LIVE (' + CONFIG['llm_model'] + ')' if llm.available else 'fallback mode (no GAPGPT_API_KEY)'}")

LLM orchestrator ready — LIVE (gpt-5.4)


## 8 · Notification engine — rules fire, LLM phrases, cooldowns guard

Two-stage by design: deterministic Python **rules** detect events and produce numeric facts; the LLM
only converts facts into natural bilingual copy (template fallback keeps it working key-less). Both
language renderings are generated **at creation time**, so switching the app language never
re-triggers API calls.

| Rule | Fires when | Cooldown key (at most once per…) |
|---|---|---|
| `budget_risk` | overrun probability ≥ 60% and ≥ 5 days into the month | month |
| `low_balance` | balance runway < 7 days of average burn | ISO week |
| `category_spike` | weekly category z-score ≥ 2 | category × month |
| `large_transaction` | single spend > 1.5 × user's 90-day P95 | transaction |
| `btc_signal` | model signal is UP or DOWN (user opted in) | signal × ISO week |
| `weekly_digest` | user-local digest day (Friday in Iran, Sunday in US) | ISO week |

Evaluation runs after every transaction write, on login, and on the background sweep — a notification
center that spams is worse than none, hence the cooldown registry lives inside `notifications.json` itself.

In [11]:
class NotificationEngine:

    def _template(self, ntype, facts, profile):
        cur, cl = profile["currency"], CATEGORY_LABELS
        if ntype == "budget_risk":
            p = facts
            return {"title": {"en": "Budget overrun risk", "fa": "خطر عبور از بودجه"},
                    "body": {"en": f"Projected month-end spend {fmt_money(p['projected_month_end'], cur, 'en')} "
                                   f"vs budget {fmt_money(p['budget'], cur, 'en')} "
                                   f"({p['overrun_probability']*100:.0f}% overrun probability).",
                             "fa": f"هزینهٔ پیش‌بینی‌شدهٔ پایان ماه {fmt_money(p['projected_month_end'], cur, 'fa')} "
                                   f"در برابر بودجهٔ {fmt_money(p['budget'], cur, 'fa')} "
                                   f"(احتمال عبور {p['overrun_probability']*100:.0f}٪)."}}
        if ntype == "low_balance":
            return {"title": {"en": "Balance running low", "fa": "موجودی رو به اتمام"},
                    "body": {"en": f"Your balance covers about {facts['runway_days']} days at the "
                                   "current spending pace.",
                             "fa": f"موجودی شما با روند فعلی حدود {facts['runway_days']} روز دیگر "
                                   "کفاف می‌دهد."}}
        if ntype == "category_spike":
            c = cl.get(facts["category"], cl["other"])
            return {"title": {"en": f"Unusual {c['en']} spending", "fa": f"هزینهٔ غیرعادی {c['fa']}"},
                    "body": {"en": f"{c['en']} hit {fmt_money(facts['this_week'], cur, 'en')} this week "
                                   f"vs a typical {fmt_money(facts['baseline_week'], cur, 'en')}.",
                             "fa": f"{c['fa']} این هفته به {fmt_money(facts['this_week'], cur, 'fa')} رسید؛ "
                                   f"میانگین معمول {fmt_money(facts['baseline_week'], cur, 'fa')} است."}}
        if ntype == "large_transaction":
            return {"title": {"en": "Large purchase detected", "fa": "خرید بزرگ ثبت شد"},
                    "body": {"en": f"{fmt_money(facts['amount'], cur, 'en')} at "
                                   f"{facts.get('payee') or 'unknown payee'} — well above your usual size.",
                             "fa": f"{fmt_money(facts['amount'], cur, 'fa')} از "
                                   f"{facts.get('payee') or 'پذیرندهٔ نامشخص'} — بسیار بالاتر از حد معمول شما."}}
        if ntype == "btc_signal":
            s = facts["signal"]
            arrow = {"UP": {"en": "upward", "fa": "صعودی"}, "DOWN": {"en": "downward", "fa": "نزولی"}}[s["signal"]]
            return {"title": {"en": "Bitcoin signal update", "fa": "به‌روزرسانی سیگنال بیت‌کوین"},
                    "body": {"en": f"The model sees a {arrow['en']} 10-day trend "
                                   f"({s['expected_move_pct']:+.1f}% expected, ±{s['uncertainty_pct']:.1f}%, "
                                   f"confidence {s['confidence']:.0%}). Not financial advice.",
                             "fa": f"مدل روند ده‌روزهٔ {arrow['fa']} پیش‌بینی می‌کند "
                                   f"({s['expected_move_pct']:+.1f}٪ با عدم قطعیت ±{s['uncertainty_pct']:.1f}٪). "
                                   "این توصیهٔ مالی نیست."}}
        if ntype == "weekly_digest":
            return {"title": {"en": "Your weekly money digest", "fa": "خلاصهٔ هفتگی مالی شما"},
                    "body": {"en": f"You spent {fmt_money(facts['week_spent'], cur, 'en')} this week"
                                   + (f", mostly on {cl[facts['top_category']]['en']}" if facts.get("top_category") else "")
                                   + f". Balance: {fmt_money(facts['balance'], cur, 'en')}.",
                             "fa": f"این هفته {fmt_money(facts['week_spent'], cur, 'fa')} هزینه کردید"
                                   + (f"، بیشتر برای {cl[facts['top_category']]['fa']}" if facts.get("top_category") else "")
                                   + f". موجودی: {fmt_money(facts['balance'], cur, 'fa')}."}}
        return {"title": {"en": "Notification", "fa": "اعلان"}, "body": {"en": "", "fa": ""}}

    def _create(self, user_id, profile, ntype, severity, facts, cooldown_key):
        doc = read_json(notif_path(user_id), {"schema_version": 1, "notifications": []})
        if any(n.get("cooldown_key") == cooldown_key for n in doc["notifications"]):
            return None                                            # cooldown: already notified
        text = llm.phrase_notification(ntype, facts, profile) or self._template(ntype, facts, profile)
        n = {"notification_id": "nt_" + uuid.uuid4().hex[:8],
             "created_at": user_now(profile).isoformat(timespec="seconds"),
             "type": ntype, "severity": severity,
             "title": text["title"], "body": text["body"],
             "facts": json.loads(json.dumps(facts, default=str)),
             "read": False, "cooldown_key": cooldown_key}
        doc["notifications"].insert(0, n)
        doc["notifications"] = doc["notifications"][:200]
        write_json_atomic(notif_path(user_id), doc)
        return n

    def evaluate(self, user_id, trigger="tx"):
        profile = accounts.get_profile(user_id)
        if not profile or not profile["preferences"].get("notifications_enabled", True):
            return []
        created, now = [], user_now(profile)
        month, iso = now.strftime("%Y-%m"), now.isocalendar()
        week_key = f"{iso.year}-W{iso.week:02d}"

        st = advisor.status(user_id, profile)
        proj = st["projection"]
        if st["overrun_risk"] == "high" and proj["days_elapsed"] >= 5:
            n = self._create(user_id, profile, "budget_risk", "critical", proj,
                             f"budget_risk:{month}")
            if n: created.append(n)
        if st["low_balance"]:
            n = self._create(user_id, profile, "low_balance", "warning",
                             {"runway_days": st["runway_days"],
                              "balance": st["balance"]["amount"]},
                             f"low_balance:{week_key}")
            if n: created.append(n)
        for a in analysis.anomalies(user_id, profile):
            if a["kind"] == "category_spike":
                n = self._create(user_id, profile, "category_spike", "warning", a,
                                 f"spike:{a['category']}:{month}")
            else:
                n = self._create(user_id, profile, "large_transaction", "info", a,
                                 f"large:{a['tx_ids'][0]}")
            if n: created.append(n)
        if profile["preferences"].get("btc_alerts", True) and btc.available:
            fc = btc.forecast()
            if fc and fc["signal"]["signal"] in ("UP", "DOWN"):
                n = self._create(user_id, profile, "btc_signal", "opportunity",
                                 {"signal": fc["signal"], "as_of": fc["as_of"],
                                  "last_close": fc["last_close"]},
                                 f"btc:{fc['signal']['signal']}:{week_key}")
                if n: created.append(n)
        if trigger == "sweep" and now.weekday() == COUNTRY_META[profile["country"]]["digest_dow"]:
            series = analysis.daily_series(user_id, profile, 7)
            week_spent = round(sum(v["total"] for v in series.values()), 2)
            by_cat = defaultdict(float)
            for v in series.values():
                for c, amt in v["by_category"].items():
                    by_cat[c] += amt
            top = max(by_cat, key=by_cat.get) if by_cat else None
            n = self._create(user_id, profile, "weekly_digest", "info",
                             {"week_spent": week_spent, "top_category": top,
                              "balance": profile["balance"]["amount"]},
                             f"digest:{week_key}")
            if n: created.append(n)
        return created

    def list(self, user_id):
        return read_json(notif_path(user_id), {"notifications": []})["notifications"]

    def unread_count(self, user_id):
        return sum(1 for n in self.list(user_id) if not n["read"])

    def mark_read(self, user_id, ids=None):
        doc = read_json(notif_path(user_id), {"schema_version": 1, "notifications": []})
        for n in doc["notifications"]:
            if ids is None or n["notification_id"] in ids:
                n["read"] = True
        write_json_atomic(notif_path(user_id), doc)
        return sum(1 for n in doc["notifications"] if not n["read"])


notifier = NotificationEngine()
print("notification engine ready — 6 rules, cooldown registry, bilingual LLM/template phrasing")

notification engine ready — 6 rules, cooldown registry, bilingual LLM/template phrasing


## 9 · REST API — one envelope, bilingual errors, whitelisted static files

Every endpoint returns `{ok, data, error}`; errors carry a machine `code` plus `message_en` /
`message_fa` so the frontend can toast in the active language without a lookup table of its own.
Routes are plain `def` (FastAPI runs them in a threadpool — correct for this file-based, lock-guarded
storage layer). Static serving is a **whitelist of exactly the five frontend files** — `data/`,
`datasets/` and the notebooks are never exposed.

In [12]:
app = FastAPI(title="AI Personal Finance Coach", docs_url=None, redoc_url=None)
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


def ok_resp(data=None):
    return JSONResponse({"ok": True, "data": data, "error": None})


def err_resp(code, status=400, detail=None):
    e = {"code": code, "message_en": ERRORS.get(code, ERRORS["server_error"])["en"],
         "message_fa": ERRORS.get(code, ERRORS["server_error"])["fa"]}
    if detail:
        e["detail"] = str(detail)[:300]
    return JSONResponse({"ok": False, "data": None, "error": e}, status_code=status)


def auth_user(token):
    uid = sessions.user_id(token)
    if not uid:
        return None, None
    return uid, accounts.get_profile(uid)


@app.exception_handler(Exception)
async def unhandled(request: Request, exc: Exception):
    return err_resp("server_error", 500, detail=repr(exc))


def _evaluate_async(user_id, trigger):
    threading.Thread(target=notifier.evaluate, args=(user_id, trigger), daemon=True).start()


@app.post("/api/auth/register")
def api_register(body: dict):
    user, err = accounts.register(body.get("username", ""), body.get("password", ""),
                                  body.get("country", ""))
    if err:
        return err_resp(err, 409 if err != "invalid_input" else 400)
    token = sessions.create(user["user_id"])
    return ok_resp({"token": token, "profile": accounts.get_profile(user["user_id"])})


@app.post("/api/auth/login")
def api_login(body: dict):
    user, err = accounts.login(body.get("username", ""), body.get("password", ""),
                               body.get("country", ""))
    if err:
        return err_resp(err, 401)
    profile = accounts.get_profile(user["user_id"])
    store.catch_up_summaries(user["user_id"], profile)
    _evaluate_async(user["user_id"], "login")
    return ok_resp({"token": sessions.create(user["user_id"]), "profile": profile})


@app.post("/api/auth/logout")
def api_logout(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    sessions.destroy(x_session_token)
    return ok_resp({"logged_out": True})


@app.get("/api/profile")
def api_profile(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp(profile)


@app.put("/api/profile")
def api_profile_update(body: dict,
                       x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    now_iso = user_now(profile).isoformat(timespec="seconds")
    if body.get("language") in ("en", "fa"):
        profile["language"] = body["language"]
    if isinstance(body.get("balance"), (int, float)) and 0 <= body["balance"] <= 1e15:
        profile["balance"] = {"amount": round(float(body["balance"]), 2),
                              "currency": profile["currency"], "updated_at": now_iso}
    if isinstance(body.get("monthly_budget"), (int, float)) and 0 <= body["monthly_budget"] <= 1e15:
        profile["monthly_budget"] = {"amount": round(float(body["monthly_budget"]), 2),
                                     "currency": profile["currency"]}
    if isinstance(body.get("category_budgets"), dict):
        profile["category_budgets"] = {c: round(float(v), 2)
                                       for c, v in body["category_budgets"].items()
                                       if c in CATEGORIES and isinstance(v, (int, float)) and v >= 0}
    if isinstance(body.get("preferences"), dict):
        for k in ("notifications_enabled", "btc_alerts"):
            if isinstance(body["preferences"].get(k), bool):
                profile["preferences"][k] = body["preferences"][k]
        if body["preferences"].get("currency_display") in ("TOMAN", "RIAL", "USD"):
            profile["preferences"]["currency_display"] = body["preferences"]["currency_display"]
    accounts.save_profile(uid, profile)
    return ok_resp(profile)

@app.post("/api/tx")
def api_tx_add(body: dict, x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    tx, err = store.add(uid, profile, body)
    if err:
        return err_resp(err)
    _evaluate_async(uid, "tx")
    return ok_resp({"transaction": tx, "balance": accounts.get_profile(uid)["balance"]})


@app.get("/api/tx")
def api_tx_list(start: str = None, end: str = None, category: str = None,
                query: str = None, limit: int = 200,
                x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    try:
        return ok_resp({"transactions": store.list(uid, profile, start, end, category, query, limit)})
    except ValueError:
        return err_resp("invalid_input")


@app.delete("/api/tx/{iso_date}/{tx_id}")
def api_tx_delete(iso_date: str, tx_id: str,
                  x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    err = store.delete(uid, profile, iso_date, tx_id)
    return err_resp(err, 404) if err else ok_resp({"deleted": tx_id})


@app.get("/api/summary/daily")
def api_daily_summary(date_iso: str = None,
                      x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    d = date_iso or user_now(profile).date().isoformat()
    existing = read_json(store.summary_path(uid, d))
    if existing and d < user_now(profile).date().isoformat():
        return ok_resp(existing)
    mode = "preview" if d == user_now(profile).date().isoformat() else "catchup"
    return ok_resp(store.build_summary(uid, profile, d, mode=mode))


@app.post("/api/ocr")
def api_ocr(file: UploadFile = File(...),
            x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    if not ocr.available:
        return err_resp("ocr_unavailable", 503)
    blob = file.file.read()
    if len(blob) > CONFIG["max_upload_mb"] * 1024 * 1024:
        return err_resp("file_too_large", 413)
    try:
        return ok_resp(ocr.process(uid, profile, blob, file.filename))
    except Exception as exc:
        return err_resp("server_error", 500, detail=exc)


@app.post("/api/stt")
def api_stt(file: UploadFile = File(...),
            x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    if not llm.available:
        return err_resp("llm_unavailable", 503)
    blob = file.file.read()
    if len(blob) > CONFIG["max_upload_mb"] * 1024 * 1024:
        return err_resp("file_too_large", 413)
    try:
        return ok_resp({"text": llm.transcribe(blob, file.filename, profile["language"])})
    except Exception as exc:
        return err_resp("server_error", 500, detail=exc)


@app.post("/api/chat")
def api_chat(body: dict, x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    message = str(body.get("message", "")).strip()
    if not message:
        return err_resp("invalid_input")

    def ndjson():
        for event in llm.chat_stream(uid, profile, message, body.get("history")):
            yield json.dumps(event, ensure_ascii=False) + "\n"

    return StreamingResponse(ndjson(), media_type="application/x-ndjson")


@app.get("/api/insights")
def api_insights(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp({"insights": analysis.insights(uid, profile)})


@app.get("/api/report")
def api_report(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    mtd = analysis.month_to_date(uid, profile)
    st = advisor.status(uid, profile)
    total = mtd["spent"] or 1.0
    return ok_resp({
        "month": mtd["month"], "currency": profile["currency"],
        "categorization": [{"category": c, "label": CATEGORY_LABELS[c],
                            "amount": v, "share_pct": round(v / total * 100, 1)}
                           for c, v in mtd["by_category"].items()],
        "spent_to_date": mtd["spent"], "income_to_date": mtd["income"],
        "projection": st["projection"], "overrun_risk": st["overrun_risk"],
        "runway_days": st["runway_days"],
        "allocation": advisor.allocation(uid, profile),
        "savings_opportunities": advisor.savings_opportunities(uid, profile),
        "insights": analysis.insights(uid, profile),
        "health": health.compute(uid, profile),
        "daily_series_30d": [{"date": d, "total": round(v["total"], 2)}
                             for d, v in sorted(analysis.daily_series(uid, profile, 30).items())]})


@app.get("/api/health-score")
def api_health(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp(health.compute(uid, profile))


@app.post("/api/whatif")
def api_whatif(body: dict, x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp(whatif.simulate(uid, profile, body.get("adjustments", {})))


@app.get("/api/subscriptions")
def api_subscriptions(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp({"subscriptions": subscriptions.detect(uid, profile)})


@app.get("/api/notifications")
def api_notifications(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, _ = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp({"notifications": notifier.list(uid)})


@app.get("/api/notifications/count")
def api_notif_count(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, _ = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    return ok_resp({"unread": notifier.unread_count(uid)})


@app.post("/api/notifications/read")
def api_notif_read(body: dict, x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, _ = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    ids = None if body.get("all") else body.get("ids", [])
    return ok_resp({"unread": notifier.mark_read(uid, ids)})


@app.get("/api/btc/history")
def api_btc_history(days: int = 30):
    try:
        return ok_resp(btc.history(int(np.clip(days, 7, 90))))
    except Exception as exc:
        return err_resp("server_error", 500, detail=exc)


@app.get("/api/btc/forecast")
def api_btc_forecast():
    if not btc.available:
        return err_resp("btc_unavailable", 503)
    fc = btc.forecast()
    return ok_resp(fc) if fc else err_resp("btc_unavailable", 503)


@app.get("/api/export/transactions.csv")
def api_export_csv(days: int = 365,
                   x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    import csv, io
    today = user_now(profile).date()
    start = (today - timedelta(days=int(np.clip(days, 1, 1000)))).isoformat()
    rows = store.list(uid, profile, start=start, limit=100000)
    buf = io.StringIO()
    w = csv.writer(buf)
    w.writerow(["date", "time", "type", "payee", "category", "amount", "currency", "note", "tags"])
    for t in rows:
        ts = t["timestamp"]
        w.writerow([ts[:10], ts[11:16], t["type"], t.get("payee", ""), t["category"],
                    t["total"]["amount"], t["currency"], t.get("note", ""),
                    " | ".join(t.get("tags", []))])
    return Response(buf.getvalue(), media_type="text/csv",
                    headers={"Content-Disposition": "attachment; filename=fiscora_transactions.csv"})


@app.get("/api/behavior")
def api_behavior(x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    try:
        return ok_resp(finance.predict(uid, profile))
    except Exception as exc:
        return err_resp("server_error", 500, detail=exc)


@app.get("/api/status")
def api_status():
    return ok_resp({"capabilities": {**CAPS, "llm_configured": llm.available,
                                     "btc_model_loaded": btc.available,
                                     "ocr_ready": ocr.available},
                    "server_time_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")})


@app.post("/api/dev/seed")
def api_seed(body: dict = None, x_session_token: str = Header(default=None, alias="X-Session-Token")):
    uid, profile = auth_user(x_session_token)
    if not uid:
        return err_resp("auth_required", 401)
    result = seed_demo_data(uid, profile, days=int((body or {}).get("days", 60)))
    _evaluate_async(uid, "tx")
    return ok_resp(result)


import mimetypes


@app.get("/api/branding")
def api_branding():
    logo = CONFIG.get("logo_path", "")
    has_logo = bool(logo and os.path.exists(logo))
    return ok_resp({"app_name": CONFIG.get("app_name", "Fiscora"),
                    "logo_url": "/branding/logo" if has_logo else None})


@app.get("/branding/logo")
def serve_logo():
    logo = CONFIG.get("logo_path", "")
    if logo and os.path.exists(logo):
        media = mimetypes.guess_type(logo)[0] or "application/octet-stream"
        return FileResponse(logo, media_type=media)
    return Response(status_code=404)


FRONTEND_FILES = {"index.html": "text/html", "AI.html": "text/html", "bitcoin.html": "text/html",
                  "notification.html": "text/html", "style.css": "text/css"}


def _serve_frontend(name):
    path = os.path.join(CONFIG["base_dir"], name)
    if os.path.exists(path):
        return FileResponse(path, media_type=FRONTEND_FILES[name])
    return Response(f"<html><body style='font-family:sans-serif;background:#101418;color:#dde'>"
                    f"<h2>{name} is not built yet</h2><p>The API is live at /api/status.</p>"
                    "</body></html>", media_type="text/html", status_code=404)


@app.get("/")
def serve_root():
    return _serve_frontend("index.html")


@app.get("/favicon")
def serve_favicon():
    return Response(status_code=204)


for _fname in FRONTEND_FILES:
    app.add_api_route(f"/{_fname}", (lambda f: (lambda: _serve_frontend(f)))(_fname),
                      methods=["GET"], name=f"static_{_fname}")

print(f"FastAPI app assembled — {len(app.routes)} routes registered")

FastAPI app assembled — 36 routes registered


In [13]:
SEED_POOLS = {
    "IR": {"salary": ("حقوق ماهانه", 45_000_000),
           "balance": 42_000_000, "budget": 30_000_000,
           "subscriptions": [("فیلیمو", "subscriptions", 159_000, 3),
                             ("اشتراک اسنپ‌پرو", "subscriptions", 99_000, 17)],
           "weekly": [("افق کوروش", "groceries", 800_000, 2_500_000),
                      ("میوه‌فروشی محله", "groceries", 300_000, 900_000)],
           "daily": [("اسنپ", "transport", 60_000, 250_000, 0.55),
                     ("اسنپ‌فود", "food_dining", 180_000, 600_000, 0.35),
                     ("کافه لمیز", "food_dining", 120_000, 350_000, 0.20),
                     ("نانوایی سنگک", "food_dining", 30_000, 90_000, 0.40),
                     ("دیجی‌کالا", "shopping", 400_000, 4_000_000, 0.10),
                     ("داروخانه شبانه‌روزی", "health", 150_000, 1_200_000, 0.06),
                     ("سینما آزادی", "entertainment", 150_000, 500_000, 0.06),
                     ("کارت به کارت به علی", "transfers", 500_000, 5_000_000, 0.05)]},
    "US": {"salary": ("Payroll deposit", 4_800),
           "balance": 5_200, "budget": 3_200,
           "subscriptions": [("Netflix", "subscriptions", 15.49, 3),
                             ("Spotify", "subscriptions", 11.99, 17)],
           "weekly": [("Walmart", "groceries", 60, 180),
                      ("Trader Joe's", "groceries", 30, 90)],
           "daily": [("Uber", "transport", 8, 35, 0.40),
                     ("Starbucks", "food_dining", 5, 14, 0.45),
                     ("Chipotle", "food_dining", 11, 28, 0.30),
                     ("DoorDash", "food_dining", 18, 55, 0.20),
                     ("Amazon", "shopping", 15, 220, 0.15),
                     ("CVS Pharmacy", "health", 8, 90, 0.07),
                     ("AMC Cinema", "entertainment", 14, 45, 0.06),
                     ("Zelle to Sarah", "transfers", 20, 300, 0.05)]},
}


def seed_demo_data(user_id, profile, days=92):
    """Replaces the user's transaction history with a realistic seeded one.
    Default 92 days ≈ 3 full months, so monthly subscriptions occur ≥3 times —
    enough for the subscription detector to lock onto their cadence."""
    rng = random.Random(99)
    pool = SEED_POOLS[profile["country"]]
    for d in (tx_dir(user_id), summary_dir(user_id)):
        if os.path.isdir(d):
            for f in os.listdir(d):
                if f.endswith(".json"):
                    os.unlink(os.path.join(d, f))
    write_json_atomic(notif_path(user_id), {"schema_version": 1, "notifications": []})

    tz = ZoneInfo(COUNTRY_META[profile["country"]]["timezone"])
    today = datetime.now(tz).date()
    count = 0
    for back in range(days, -1, -1):
        d = today - timedelta(days=back)
        def add(payee, category, amount, hour, tx_type="purchase"):
            nonlocal count
            ts = datetime(d.year, d.month, d.day, hour, rng.randint(0, 59), tzinfo=tz)
            _, err = store.add(user_id, profile, {
                "payee": payee, "amount": round(amount, 2), "type": tx_type,
                "category": category, "timestamp": ts.isoformat(), "source": "seeded"})
            count += 0 if err else 1
        if d.day == 1:
            add(pool["salary"][0], "income", pool["salary"][1], 9, tx_type="income")
        for payee, cat, amt, dom in pool["subscriptions"]:
            if d.day == dom:
                add(payee, cat, amt, 8, tx_type="subscription")
        if d.weekday() == COUNTRY_META[profile["country"]]["week_start_dow"]:
            payee, cat, lo, hi = pool["weekly"][0]                 # fixed weekly cadence (detectable)
            add(payee, cat, rng.uniform(lo, hi), 18)
            if rng.random() < 0.5:
                payee, cat, lo, hi = pool["weekly"][1]
                add(payee, cat, rng.uniform(lo, hi), 19)
        for payee, cat, lo, hi, prob in pool["daily"]:
            recency_boost = 1.35 if back <= 12 and cat == "food_dining" else 1.0   # a visible trend
            if rng.random() < prob * recency_boost:
                add(payee, cat, rng.uniform(lo, hi), rng.randint(8, 22))

    profile = accounts.get_profile(user_id)
    now_iso = user_now(profile).isoformat(timespec="seconds")
    profile["balance"] = {"amount": pool["balance"], "currency": profile["currency"],
                          "updated_at": now_iso}
    profile["monthly_budget"] = {"amount": pool["budget"], "currency": profile["currency"]}
    accounts.save_profile(user_id, profile)
    created = store.catch_up_summaries(user_id, profile)
    return {"seeded_transactions": count, "seeded_days": days, "summaries_created": created,
            "balance": profile["balance"], "monthly_budget": profile["monthly_budget"]}


print("demo seeder ready — POST /api/dev/seed builds a realistic 92-day history for the logged-in user")

demo seeder ready — POST /api/dev/seed builds a realistic 92-day history for the logged-in user


## 10 · LAUNCH — notebook-safe server + background scheduler

**Why this never hits `asyncio.run() cannot be called from a running event loop`:** Jupyter's kernel
owns the main thread's event loop, so uvicorn is started inside a **dedicated daemon thread that
creates its own loop** (`asyncio.new_event_loop()` → `loop.run_until_complete(server.serve())`).
Jupyter's loop and uvicorn's loop never touch. Re-running the launch cell performs a clean restart:
signals the old server to exit, joins the thread, starts a fresh one.

The **scheduler thread** (same idempotent pattern) wakes every 60 s and:
- writes the **23:59 daily summary** for each user in *their* timezone (and back-fills any days the
  notebook was off, marked `generation_mode: "catchup"`),
- runs the notification **sweep** every 30 min (weekly digests fire here, on the user-local digest day),
- refreshes the BTC price cache hourly.

In [14]:
class _ServerThread(threading.Thread):
    def __init__(self, asgi_app, port):
        super().__init__(daemon=True, name="pfc-uvicorn")
        self._config = uvicorn.Config(asgi_app, host="127.0.0.1", port=port,
                                      log_level="warning", loop="asyncio")
        self.server = uvicorn.Server(self._config)

    def run(self):
        import asyncio
        loop = asyncio.new_event_loop()          # own loop — never Jupyter's
        asyncio.set_event_loop(loop)
        loop.run_until_complete(self.server.serve())

    def stop(self):
        self.server.should_exit = True


if "_SERVER_THREAD" in globals() and _SERVER_THREAD.is_alive():   # idempotent restart
    _SERVER_THREAD.stop()
    _SERVER_THREAD.join(timeout=8)
    print("previous server instance stopped")

_SERVER_THREAD = _ServerThread(app, CONFIG["port"])
_SERVER_THREAD.start()

BASE_URL = f"http://127.0.0.1:{CONFIG['port']}"
for _ in range(60):                              # readiness gate
    try:
        requests.get(BASE_URL + "/api/status", timeout=1)
        break
    except requests.RequestException:
        time.sleep(0.25)
else:
    raise RuntimeError(f"server failed to start on port {CONFIG['port']} — is the port in use?")

caps = requests.get(BASE_URL + "/api/status", timeout=3).json()["data"]["capabilities"]
print(f"✅  AI Personal Finance Coach is LIVE →  {BASE_URL}")
print(f"    llm={'on' if caps['llm_configured'] else 'fallback'} · "
      f"btc_model={'loaded' if caps['btc_model_loaded'] else 'missing'} · "
      f"ocr={'ready' if caps['ocr_ready'] else 'not installed'}")

✅  AI Personal Finance Coach is LIVE →  http://127.0.0.1:5050
    llm=on · btc_model=loaded · ocr=ready


In [15]:
if "_SCHEDULER_STOP" in globals():
    _SCHEDULER_STOP.set()                        # idempotent restart
_SCHEDULER_STOP = threading.Event()


def _scheduler_loop(stop_event):
    last_sweep = last_btc = 0.0
    while not stop_event.is_set():
        try:
            for uid in accounts.all_user_ids():
                profile = accounts.get_profile(uid)
                if not profile:
                    continue
                store.catch_up_summaries(uid, profile)
                now = user_now(profile)
                if now.hour == 23 and now.minute == 59:            # the 23:59 snapshot, user-local
                    store.build_summary(uid, profile, now.date().isoformat(), mode="scheduled")
            if time.time() - last_sweep >= CONFIG["sweep_interval_s"]:
                for uid in accounts.all_user_ids():
                    notifier.evaluate(uid, trigger="sweep")
                last_sweep = time.time()
            if time.time() - last_btc >= CONFIG["btc_refresh_s"]:
                btc.get_rows(force_refresh=True)
                last_btc = time.time()
        except Exception as exc:                                   # scheduler must never die
            print(f"[scheduler] recovered from: {exc!r}")
        stop_event.wait(60)


_SCHEDULER_THREAD = threading.Thread(target=_scheduler_loop, args=(_SCHEDULER_STOP,),
                                     daemon=True, name="pfc-scheduler")
_SCHEDULER_THREAD.start()
print("scheduler running — daily 23:59 summaries (+catch-up), 30-min notification sweep, hourly BTC refresh")

scheduler running — daily 23:59 summaries (+catch-up), 30-min notification sweep, hourly BTC refresh


In [16]:
def run_smoke_tests():
    S, results = requests.Session(), []
    username, password = f"demo_{int(time.time()) % 100000}", "Demo!234"

    def check(name, cond, note=""):
        results.append((name, bool(cond), note))

    r = S.post(BASE_URL + "/api/auth/register",
               json={"username": username, "password": password, "country": "IR"}).json()
    check("register (IR)", r["ok"])
    S.headers["X-Session-Token"] = r["data"]["token"]

    r2 = S.post(BASE_URL + "/api/auth/register",
                json={"username": username, "password": password, "country": "IR"}).json()
    check("duplicate register rejected", not r2["ok"] and r2["error"]["code"] == "account_exists_login")
    r2 = S.post(BASE_URL + "/api/auth/login",
                json={"username": username, "password": "wrong", "country": "IR"}).json()
    check("wrong password rejected", not r2["ok"])
    r2 = S.post(BASE_URL + "/api/auth/login",
                json={"username": username, "password": password, "country": "IR"}).json()
    check("login (triple match)", r2["ok"])

    r = S.put(BASE_URL + "/api/profile", json={"balance": 42_000_000,
                                               "monthly_budget": 30_000_000}).json()
    check("profile update", r["ok"] and r["data"]["balance"]["amount"] == 42_000_000)

    r = S.post(BASE_URL + "/api/tx", json={"payee": "اسنپ‌فود", "amount": 350_000,
                                           "note": "شام"}).json()
    check("manual tx + auto-category (fa)", r["ok"] and
          r["data"]["transaction"]["category"] == "food_dining")
    check("balance auto-adjusted", r["data"]["balance"]["amount"] == 42_000_000 - 350_000)
    bad = S.post(BASE_URL + "/api/tx", json={"payee": "x", "amount": -5}).json()
    check("invalid amount rejected", not bad["ok"])

    r = S.post(BASE_URL + "/api/dev/seed", json={"days": 92}).json()
    check("seed 92-day demo history", r["ok"] and r["data"]["seeded_transactions"] > 100,
          f"{r['data']['seeded_transactions']} txs" if r["ok"] else "")

    r = S.get(BASE_URL + "/api/tx", params={"limit": 5}).json()
    check("tx list", r["ok"] and len(r["data"]["transactions"]) == 5)
    r = S.get(BASE_URL + "/api/report").json()
    check("report", r["ok"] and r["data"]["categorization"] and r["data"]["projection"])
    r = S.get(BASE_URL + "/api/insights").json()
    check("insights", r["ok"], f"{len(r['data']['insights'])} insights" if r["ok"] else "")
    r = S.get(BASE_URL + "/api/health-score").json()
    check("health score", r["ok"] and 0 <= r["data"]["score"] <= 100,
          f"score {r['data']['score']}" if r["ok"] else "")
    r = S.post(BASE_URL + "/api/whatif", json={"adjustments": {"food_dining": -30}}).json()
    check("what-if simulator", r["ok"] and r["data"]["monthly_saving"] >= 0)
    r = S.get(BASE_URL + "/api/subscriptions").json()
    check("subscription detector", r["ok"] and len(r["data"]["subscriptions"]) >= 1,
          f"{len(r['data']['subscriptions'])} found" if r["ok"] else "")
    r = S.get(BASE_URL + "/api/summary/daily").json()
    check("daily summary", r["ok"] and r["data"]["file_type"] == "daily_summary")

    time.sleep(1.5)
    r = S.get(BASE_URL + "/api/notifications").json()
    check("notifications", r["ok"], f"{len(r['data']['notifications'])} created" if r["ok"] else "")
    r = S.get(BASE_URL + "/api/notifications/count").json()
    check("unread count", r["ok"])
    r = S.post(BASE_URL + "/api/notifications/read", json={"all": True}).json()
    check("mark all read", r["ok"] and r["data"]["unread"] == 0)

    with S.post(BASE_URL + "/api/chat", json={"message": "How much did I spend last month?"},
                stream=True) as resp:
        events = [json.loads(l) for l in resp.iter_lines() if l]
    kinds = {e["type"] for e in events}
    check("chat stream (searching→tokens→done)", "token" in kinds and "done" in kinds,
          "live LLM" if llm.available else "fallback mode")

    r = S.get(BASE_URL + "/api/btc/history").json()
    check("btc history", r["ok"] and len(r["data"]["days"]) == 30,
          r["data"]["source"] if r["ok"] else "")
    r = S.get(BASE_URL + "/api/btc/forecast").json()
    check("btc forecast", r["ok"] or r["error"]["code"] == "btc_unavailable",
          r["data"]["signal"]["signal"] if r["ok"] else "model not trained yet — run bitcoin.ipynb")

    png = base64.b64decode(
        "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR4nGP4z8DwHwAFAAH/q842iQAAAABJRU5ErkJggg==")
    r = S.post(BASE_URL + "/api/ocr", files={"file": ("t.png", png, "image/png")}).json()
    check("ocr endpoint", r["ok"] or r["error"]["code"] == "ocr_unavailable",
          "engine ready" if r["ok"] else "graceful: easyocr not installed")
    r = S.post(BASE_URL + "/api/stt", files={"file": ("t.webm", b"\x1aE\xdf\xa3", "audio/webm")}).json()
    check("stt endpoint", r["ok"] or r["error"]["code"] in ("llm_unavailable", "server_error"),
          "live" if r["ok"] else "graceful: no API key / bad audio")

    r = requests.get(BASE_URL + "/api/profile").json()
    check("auth required without token", not r["ok"] and r["error"]["code"] == "auth_required")

    width = max(len(n) for n, _, _ in results)
    print("\n=== SMOKE TEST RESULTS ===")
    failed = 0
    for name, passed, note in results:
        failed += (not passed)
        print(f"  {'✅' if passed else '❌'}  {name:<{width}}  {note}")
    print(f"\n{'ALL ' + str(len(results)) + ' CHECKS PASSED' if not failed else str(failed) + ' CHECK(S) FAILED'}")
    print(f"demo account for the UI → username: {username} · password: {password} · country: IR")
    return failed == 0


run_smoke_tests()


=== SMOKE TEST RESULTS ===
  ✅  register (IR)                        
  ✅  duplicate register rejected          
  ✅  wrong password rejected              
  ✅  login (triple match)                 
  ✅  profile update                       
  ✅  manual tx + auto-category (fa)       
  ✅  balance auto-adjusted                
  ✅  invalid amount rejected              
  ✅  seed 92-day demo history             213 txs
  ✅  tx list                              
  ✅  report                               
  ✅  insights                             8 insights
  ✅  health score                         score 70.9
  ✅  what-if simulator                    
  ✅  subscription detector                2 found
  ✅  daily summary                        
  ✅  notifications                        0 created
  ✅  unread count                         
  ✅  mark all read                        
  ✅  chat stream (searching→tokens→done)  live LLM
  ✅  btc history                          coingecko_approx
  

/Users/sam/Desktop/aiolearn/venv/lib/python3.12/site-packages/torch/ao/nn/quantized/dynamic/modules/rnn.py:162: UserWarning: torch.quantize_per_tensor, torch.quantize_per_channel and other quantized tensor creation functions that produce tensors with dtype torch.quint8, torch.qint8, and torch.qint32 are deprecated and will be removed in a future PyTorch release. Please see https://github.com/pytorch/pytorch/issues/184982 for more information. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/quantized/Quantizer.cpp:116.)
  w_ih = torch.quantize_per_tensor(


True

## 11 · Operations notes

**Run order.** `bitcoin.ipynb` once (produces `data/btc/*`), then this notebook top-to-bottom, then
open **http://localhost:5050**. Keep the kernel alive; re-run the LAUNCH cell to restart the server
after a code change.

**Configuration.**
- `GAPGPT_API_KEY` — set as an environment variable before starting Jupyter (or in the inline
  override in cell 1 for throwaway demos). Without it: chat answers from the deterministic fallback,
  notifications use templates, OCR uses regex extraction — nothing breaks.
- OCR engine: `pip install easyocr` (~1 GB of PyTorch weights; first request warms the reader).
- Port: change `CONFIG["port"]` and re-run from cell 1.

**Security truths (demo-grade by design, stated honestly):** localhost HTTP, tokens in
`localStorage`, JSON-on-disk storage. We still do the cheap right things — salted PBKDF2 (310k
iterations), the API key never leaves the server, static serving is whitelisted to exactly five
files, uploads are size-capped, every input is validated before touching storage. Production path:
encrypted DB, HTTPS, httpOnly cookies, real session expiry.

**Data layout produced at runtime:**
```
data/
├─ users.json · sessions.json
├─ users/<user_id>/
│  ├─ profile.json · notifications.json · payee_categories.json
│  ├─ transactions/YYYY-MM-DD.json        (per-day logs)
│  ├─ summaries/YYYY-MM-DD.summary.json   (23:59 snapshots + catch-up)
│  └─ receipts/rc_*.jpg
└─ btc/ model.keras · scalers.pkl · metadata.json · price_cache.json
```